In [1]:
import numpy as np
import pygmt
import pandas as pd
import re
import matplotlib.pyplot as plt
import os
import tkinter as tk
import tkinter.font as tkFont
import locale
from datetime import datetime, timedelta
from docx import Document
from docx.shared import Pt, Inches
from docx.shared import Cm
from docx.enum.text import WD_PARAGRAPH_ALIGNMENT
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.enum.section import WD_SECTION, WD_ORIENT
from openpyxl import load_workbook
from tkcalendar import DateEntry 
from tkinter import ttk, filedialog, messagebox, scrolledtext



# ========== COLOR PALETTE ==========
COLOR_WHITE = "#FFFFFF"
COLOR_LIGHT_BG = "#F5EFE6"
COLOR_GREEN_PASTEL = "#6D94C5"
COLOR_BLUE_PASTEL = "#CBDCEB"
COLOR_DARK_TEXT = "#000000"
COLOR_BORDER = "#E0E7FF"
COLOR_SIDEBAR_BG = "#8F2C2C"
COLOR_BUTTON_BG = "#E8DFCA"
COLOR_PETUNJUK_BG = "#E8DFCA"

# ========== STATE ==========
selected_files = []

# ========== FUNCTIONS ==========
def select_files():
    """Handle file selection"""
    global selected_files
    files = filedialog.askopenfilenames(
        title="Pilih File",
        filetypes=[("All Files", "*.*"), ("Excel Files", "*.xlsx"), ("CSV Files", "*.csv")]
    )
    if files:
        selected_files = list(files)
        file_status_label.config(text=f"✅ {len(selected_files)} file(s) dipilih", fg="#2D7F2D")
    else:
        file_status_label.config(text="❌ File belum dipilih", fg="#C00000")

def process_data():
    import xml.etree.ElementTree as ET
    import pandas as pd
    from datetime import datetime, timedelta
    import os
    from tkinter import messagebox
    global selected_files, tanggal_awal, tanggal_akhir

    """Process data dengan start dan end date"""
    start = start_date_entry.get_date()
    end = end_date_entry.get_date()
    tanggal_awal = pd.to_datetime(start, format='%d/%m/%Y')
    tanggal_akhir = pd.to_datetime(end, format='%d/%m/%Y')
    
    if not selected_files:
        messagebox.showwarning("Peringatan", "⚠️ Silakan pilih file terlebih dahulu")
        return None, None
    
    ns = {'sc': 'http://geofon.gfz-potsdam.de/ns/seiscomp3-schema/0.13'}
    combined_data = pd.DataFrame()

    # Loop tiap file XML
    for file_path in selected_files:
        try:
            tree = ET.parse(file_path)
            root = tree.getroot()
            data_csv = []

            for event in root.findall(".//sc:event", ns):
                event_id = event.attrib.get("publicID")
                preferred_origin_id = event.findtext("sc:preferredOriginID", namespaces=ns)
                preferred_magnitude_id = event.findtext("sc:preferredMagnitudeID", namespaces=ns)
                deskripsi = event.find("sc:description/sc:text", ns)
                deskripsi_lokasi = deskripsi.text if deskripsi is not None else ""

                origin = root.find(f".//sc:origin[@publicID='{preferred_origin_id}']", ns)
                if origin is None:
                    continue

                # Ambil data utama
                waktu_utc = origin.findtext("sc:time/sc:value", namespaces=ns)
                latitude = float(origin.findtext("sc:latitude/sc:value", namespaces=ns))
                longitude = float(origin.findtext("sc:longitude/sc:value", namespaces=ns))
                depth = float(origin.findtext("sc:depth/sc:value", namespaces=ns))

                # Konversi waktu ke UTC+7
                t_local = datetime.strptime(waktu_utc, "%Y-%m-%dT%H:%M:%S.%fZ") + timedelta(hours=7)
                tanggal = t_local.strftime("%Y-%m-%d")
                jam = t_local.strftime("%H:%M:%S")

                # Parse magnitude dengan error handling
                magnitude = None
                if preferred_magnitude_id is not None:
                    try:
                        mag_elem = root.find(f".//sc:magnitude[@publicID='{preferred_magnitude_id}']/sc:magnitude/sc:value", ns)
                        if mag_elem is not None:
                            magnitude = round(float(mag_elem.text), 1)
                    except (ValueError, AttributeError):
                        magnitude = None

                # Simpan hasil untuk CSV
                data_csv.append({
                    "Tanggal": tanggal,
                    "Waktu (WIB)": jam,
                    "Lintang": round(latitude, 2),
                    "Bujur": round(longitude, 2),
                    "Kedalaman": int(depth),
                    "Magnitude": magnitude,
                    "Keterangan": deskripsi_lokasi,
                    "Dirasakan": ''
                })

            # Buat DataFrame dari CSV
            parsed_data = pd.DataFrame(data_csv)
            parsed_data['Tanggal'] = pd.to_datetime(parsed_data['Tanggal'], errors='coerce')

            # Filter tanggal & koordinat & magnitude
            mask = (
                (parsed_data['Tanggal'] >= tanggal_awal) &
                (parsed_data['Tanggal'] <= tanggal_akhir) &
                (parsed_data['Bujur'].astype(float) >= 113.95) &
                (parsed_data['Bujur'].astype(float) <= 117.31) &
                (parsed_data['Lintang'].astype(float) >= -13.65) &
                (parsed_data['Lintang'].astype(float) <= -6.68) 
            )
            filtered_data = parsed_data[mask].copy()
            combined_data = pd.concat([combined_data, filtered_data], ignore_index=True)

        except Exception as e:
            messagebox.showerror("Error", f"Error parsing {file_path}: {str(e)}")
            continue

    if combined_data.empty:
        messagebox.showwarning("Warning", "Tidak ada data yang sesuai filter!")
        return

    # ===== FUNGSI: DEDUPLIKASI GEMPA =====
    def deduplicate_earthquakes(df, time_tolerance_seconds=5):
        """
        Menghapus gempa duplikat berdasarkan tanggal + waktu (±5 detik)
        Menyimpan data pertama (index terkecil) jika ada duplikat
        """
        if df.empty:
            return df

        df = df.sort_values(by=['Tanggal', 'Waktu (WIB)']).reset_index(drop=True)
        df['Datetime'] = pd.to_datetime(
            df['Tanggal'].astype(str) + ' ' + df['Waktu (WIB)'],
            errors='coerce'
        )

        duplicates_to_drop = []

        for i in range(len(df)):
            if i in duplicates_to_drop:
                continue

            current_time = df.loc[i, 'Datetime']
            current_date = df.loc[i, 'Tanggal']

            # Bandingkan dengan gempa berikutnya
            for j in range(i + 1, len(df)):
                if j in duplicates_to_drop:
                    continue

                compare_time = df.loc[j, 'Datetime']
                compare_date = df.loc[j, 'Tanggal']

                # Jika tanggal berbeda, skip
                if current_date != compare_date:
                    break

                # Cek perbedaan waktu
                time_diff = abs((current_time - compare_time).total_seconds())

                if time_diff <= time_tolerance_seconds:
                    # Tandai untuk dihapus (keep yang pertama)
                    duplicates_to_drop.append(j)

        # Hapus duplikat
        df_deduplicated = df.drop(duplicates_to_drop).reset_index(drop=True)
        df_deduplicated = df_deduplicated.drop(columns=['Datetime'])

        return df_deduplicated

    # ===== FUNGSI: MERGE DENGAN DIRASAKAN =====
    def merge_with_dirasakan(combined_data, dirasakan_file, time_tolerance_seconds=5):
        """
        Merge data gempa dengan dirasakan.xlsx
        Match berdasarkan tanggal + waktu (±5 detik)
        SEMUA parameter dari Excel menggantikan parameter CSV jika ada match
        """
        if not os.path.exists(dirasakan_file):
            messagebox.showerror("Error", f"File '{dirasakan_file}' tidak ditemukan!")
            return combined_data

        existing_data = pd.read_excel(dirasakan_file)

        # Normalize kolom - CONVERT WAKTU KE STRING DULU
        existing_data['Tanggal'] = pd.to_datetime(existing_data['Tanggal'], errors='coerce')
        # Konversi Waktu (WIB) ke string jika masih datetime.time
        if existing_data['Waktu (WIB)'].dtype == 'object':
            existing_data['Waktu (WIB)'] = existing_data['Waktu (WIB)'].astype(str)
        else:
            # Jika datetime.time atau datetime64, convert ke string
            existing_data['Waktu (WIB)'] = existing_data['Waktu (WIB)'].apply(lambda x: str(x))
        
        existing_data['Datetime'] = pd.to_datetime(
            existing_data['Tanggal'].astype(str) + ' ' + existing_data['Waktu (WIB)'],
            errors='coerce'
        )

        combined_data['Datetime'] = pd.to_datetime(
            combined_data['Tanggal'].astype(str) + ' ' + combined_data['Waktu (WIB)'],
            errors='coerce'
        )

        # Tandai data yang akan diganti dengan data dari Excel
        combined_data['from_excel'] = False
        indices_to_replace = []

        # Loop: cari match di existing data
        for idx_existing, row_existing in existing_data.iterrows():
            for idx_combined, row_combined in combined_data.iterrows():
                if row_combined['from_excel']:
                    # Skip jika sudah diganti dari Excel
                    continue
                    
                time_diff = abs((row_combined['Datetime'] - row_existing['Datetime']).total_seconds())

                if time_diff <= time_tolerance_seconds:
                    # Tandai untuk diganti dengan data dari Excel
                    indices_to_replace.append((idx_combined, idx_existing))
                    combined_data.loc[idx_combined, 'from_excel'] = True
                    break

        # Ganti data CSV dengan data dari Excel
        for idx_combined, idx_existing in indices_to_replace:
            # Ambil semua kolom dari Excel (kecuali 'Datetime')
            row_excel = existing_data.loc[idx_existing].to_dict()
            
            # Update semua kolom di combined_data dengan data dari Excel
            for col in existing_data.columns:
                if col != 'Datetime' and col in combined_data.columns:
                    combined_data.loc[idx_combined, col] = row_excel[col]

        # Append gempa dari existing_data yang tidak ter-match
        for idx_existing, row_existing in existing_data.iterrows():
            matched = False
            
            for idx_combined, row_combined in combined_data.iterrows():
                if row_combined['from_excel']:
                    time_diff = abs((row_combined['Datetime'] - row_existing['Datetime']).total_seconds())
                    if time_diff <= time_tolerance_seconds:
                        matched = True
                        break

            if not matched:
                # Tambahkan data existing yang tidak ada di combined
                new_row = row_existing.drop(['Datetime']).to_dict()
                new_row['from_excel'] = True
                combined_data = pd.concat([combined_data, pd.DataFrame([new_row])], ignore_index=True)

        # Hapus kolom temporary
        combined_data = combined_data.drop(columns=['from_excel', 'Datetime'])

        return combined_data

    # ===== PROSES UTAMA =====

    # 1. Deduplikasi gempa
    combined_data = deduplicate_earthquakes(combined_data, time_tolerance_seconds=5)

    # 2. Merge dengan dirasakan.xlsx
    existing_file = 'dirasakan.xlsx'
    combined_data = merge_with_dirasakan(combined_data, existing_file, time_tolerance_seconds=5)

    # 3. Sort & format output
    combined_data['Tanggal'] = pd.to_datetime(combined_data['Tanggal'])
    combined_data['Datetime'] = pd.to_datetime(
        combined_data['Tanggal'].astype(str) + ' ' + combined_data['Waktu (WIB)'],
        errors='coerce'
    )
    combined_data = combined_data.sort_values(by='Datetime').reset_index(drop=True)
    combined_data = combined_data.drop(columns=['Datetime'])
    combined_data['Tanggal'] = combined_data['Tanggal'].dt.date

    # 4. Reorder columns & add No
    column_order = ['No', 'Tanggal', 'Waktu (WIB)', 'Lintang', 'Bujur', 'Kedalaman', 'Magnitude', 'Keterangan', 'Dirasakan']
    combined_data.insert(0, 'No', range(1, len(combined_data) + 1))
    combined_data = combined_data[column_order]

    # 5. Save output
    combined_data.to_csv('datalengkap.csv', index=False)

    # Tentukan nama folder periode
    if tanggal_awal.month == tanggal_akhir.month:
        nama_folder = f"{tanggal_awal.day:02d} - {tanggal_akhir.day:02d} {tanggal_awal.strftime('%B %Y')}"
    else:
        nama_folder = f"{tanggal_awal.day:02d} {tanggal_awal.strftime('%B')} - {tanggal_akhir.day:02d} {tanggal_akhir.strftime('%B %Y')}"

    if not os.path.exists(nama_folder):
        os.makedirs(nama_folder)

    output_file = os.path.join(nama_folder, f'Data Gempabumi Periode {nama_folder}.xlsx')
    combined_data.to_excel(output_file, index=False)

    messagebox.showinfo("Proses", f"Data berhasil diproses dan disimpan ke:\n1. {output_file}\n2. datalengkap.csv di direktori script.")

    
    # Create popup window
    popup = tk.Toplevel(root)
    popup.title("Hasil Proses Data")
    popup.geometry("600x450")
    popup.configure(bg=COLOR_WHITE)
    popup.resizable(False, False)
    
    # Header
    header_frame = tk.Frame(popup, bg=COLOR_GREEN_PASTEL, height=60)
    header_frame.pack(fill=tk.X)
    header_frame.pack_propagate(False)
    header_label = tk.Label(header_frame, text="📊 Hasil Proses Data", 
                           font=("Gill Sans MT", 16, "bold"), bg=COLOR_GREEN_PASTEL, fg=COLOR_WHITE)
    header_label.pack(pady=15)
    
    # Content
    content_frame = tk.Frame(popup, bg=COLOR_WHITE)
    content_frame.pack(fill=tk.BOTH, expand=True, padx=15, pady=15)
    
    info_text = f"""✅ PROSES DATA BERHASIL

Tanggal Awal:    {start}
Tanggal Akhir:   {end}
File diproses:   {len(selected_files)} file(s)

📁 Daftar File:
"""
    for i, file in enumerate(selected_files, 1):
        filename = file.split('/')[-1] if '/' in file else file.split('\\')[-1]
        info_text += f"  {i}. {filename}\n"
    
    info_text += f"\n⏱️ Waktu pemrosesan: {datetime.now().strftime('%H:%M:%S')}"
    
    text_widget = scrolledtext.ScrolledText(content_frame, height=18, width=65, 
                                           bg=COLOR_LIGHT_BG, fg=COLOR_DARK_TEXT,
                                           font=("Courier", 10), relief=tk.FLAT,
                                           borderwidth=0)
    text_widget.pack(fill=tk.BOTH, expand=True)
    text_widget.insert(tk.END, info_text)
    text_widget.config(state=tk.DISABLED)
    
    # Close button
    close_btn = tk.Button(popup, text="Tutup", command=popup.destroy,
                         bg=COLOR_GREEN_PASTEL, fg=COLOR_WHITE,
                         font=("Gill Sans MT", 10, "bold"), padx=30, pady=10,
                         relief=tk.FLAT, cursor="hand2", bd=0)
    close_btn.pack(pady=10)

def print_analysis():
    def read_csv_and_analyze(tanggal_awal, tanggal_akhir):
        
        # Step 1: Read CSV file
        file_path = "datalengkap.csv"
        data = pd.read_csv(file_path)
        
        # Konversi kolom Tanggal ke datetime dan ambil hanya tanggalnya
        data['Tanggal'] = pd.to_datetime(data['Tanggal']).dt.date
        tanggal_awal = pd.to_datetime(tanggal_awal).date()
        tanggal_akhir = pd.to_datetime(tanggal_akhir).date()
          
        # Filter data berdasarkan tanggal yang dipilih
        data = data[(data['Tanggal'] >= tanggal_awal) & (data['Tanggal'] <= tanggal_akhir)]
        all_dates = pd.date_range(start=tanggal_awal, end=tanggal_akhir, freq='D')
        periode = all_dates.strftime('%Y-%m-%d')  # Format tanggal sebagai string

        # Drop the 'Magnitude Category' and 'Depth Category' columns
        output_data = data.drop(columns=['Magnitude Category', 'Depth Category'], errors='ignore')

        # Create magnitude and depth categories
        categories_magnitude = ['M<3', '3≤M<5', 'M≥5']
        data['Magnitude'] = pd.to_numeric(data['Magnitude'], errors='coerce')
        data['Magnitude Category'] = pd.cut(data['Magnitude'], bins=[0, 3, 5, float('inf')], labels=categories_magnitude, right=False)

        categories_depth = ['D≤60 km', '60<D≤300 km', 'D>300 km']
        data['Kedalaman'] = pd.to_numeric(data['Kedalaman'], errors='coerce')
        data['Depth Category'] = pd.cut(data['Kedalaman'], bins=[0, 60, 300, float('inf')], labels=categories_depth, right=False)       
        
        # Group data by day and categories for analysis
        # magnitude_summary = (data.groupby([data['Tanggal'], 'Magnitude Category']).size().unstack(fill_value=0).reindex(index=all_dates, fill_value=0).reindex(columns=categories_magnitude, fill_value=0))
        # depth_summary = (data.groupby([data['Tanggal'], 'Depth Category']).size().unstack(fill_value=0).reindex(index=all_dates, fill_value=0).reindex(columns=categories_depth, fill_value=0))
        
        magnitude_summary = (data.groupby([data['Tanggal'], 'Magnitude Category'])
                            .size()
                            .unstack(fill_value=0)
                            .reindex(index=(pd.date_range(start=tanggal_awal, end=tanggal_akhir, freq='D')), fill_value=0)
                            .reindex(columns=categories_magnitude, fill_value=0))

        depth_summary = (data.groupby([data['Tanggal'], 'Depth Category'])
                        .size()
                        .unstack(fill_value=0)
                        .reindex(index=(pd.date_range(start=tanggal_awal, end=tanggal_akhir, freq='D')), fill_value=0)
                        .reindex(columns=categories_depth, fill_value=0))

        # Add totals row and columns for 'Gempa Dirasakan' and 'Gempa Merusak'
        magnitude_summary['Jumlah Total'] = magnitude_summary.sum(axis=1)
        magnitude_summary['Gempa Dirasakan'] = 0
        magnitude_summary['Gempa Merusak'] = 0
        magnitude_summary.loc['Jumlah Gempa'] = magnitude_summary.sum()

        depth_summary['Jumlah Total'] = depth_summary.sum(axis=1)
        depth_summary['Gempa Dirasakan'] = 0
        depth_summary['Gempa Merusak'] = 0
        depth_summary.loc['Jumlah Gempa'] = depth_summary.sum()

        # Mengisi kolom 'Gempa Dirasakan' jika kolom 'Dirasakan' terisi
        for index, row in data.iterrows():
            if pd.notna(row['Dirasakan']) and row['Dirasakan'].strip() != '':
                if row['Tanggal'] in magnitude_summary.index:
                    magnitude_summary.at[row['Tanggal'], 'Gempa Dirasakan'] += 1
                if row['Tanggal'] in depth_summary.index:
                    depth_summary.at[row['Tanggal'], 'Gempa Dirasakan'] += 1

        magnitude_summary.at['Jumlah Gempa', 'Gempa Dirasakan'] = magnitude_summary['Gempa Dirasakan'].sum()
        depth_summary.at['Jumlah Gempa', 'Gempa Dirasakan'] = depth_summary['Gempa Dirasakan'].sum()

        if tanggal_awal.month == tanggal_akhir.month:
            nama_folder = f"{tanggal_awal.day:02d} - {tanggal_akhir.day:02d} {tanggal_awal.strftime('%B %Y')}"
        else:
            nama_folder = f"{tanggal_awal.day:02d} {tanggal_awal.strftime('%B')} - {tanggal_akhir.day:02d} {tanggal_akhir.strftime('%B %Y')}"

        # Konversi index (tanggal) ke format date-only sebelum disimpan ke Excel
        # Untuk menangani row 'Jumlah Gempa' yang bukan tanggal
        magnitude_summary.index = [x.strftime('%Y-%m-%d') if isinstance(x, pd.Timestamp) else x for x in magnitude_summary.index]
        depth_summary.index = [x.strftime('%Y-%m-%d') if isinstance(x, pd.Timestamp) else x for x in depth_summary.index]

        # Save the analysis to Excel
        output_file = os.path.join(nama_folder, f'Analisis Gempabumi Periode {nama_folder}.xlsx')
        with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
            output_data.to_excel(writer, sheet_name='Earthquake Data', index=False)
            magnitude_summary.to_excel(writer, sheet_name='Magnitude Summary')
            depth_summary.to_excel(writer, sheet_name='Depth Summary')
            plot_charts(magnitude_summary, depth_summary, writer, nama_folder, tanggal_awal, tanggal_akhir)

        # Autofit columns
        autofit_columns(output_file, 'Earthquake Data')
        autofit_columns(output_file, 'Magnitude Summary')
        autofit_columns(output_file, 'Depth Summary')

        messagebox.showinfo("Info", f"Analisis berhasil disimpan di:\n{output_file}")

    def plot_charts(magnitude_summary, depth_summary, writer, nama_folder, tanggal_awal, tanggal_akhir,):
        # Set the size for both bar and pie charts
        bar_chart_size = (16, 10)
        pie_chart_size = (5, 5)

        # ================== MAGNITUDE BAR CHART ==================
        categories = ['M<3', '3≤M<5', 'M≥5']
        colors = ['green', 'yellow', 'red']
        
        # Remove total row
        magnitude_summary_without_total = magnitude_summary.drop(index='Jumlah Gempa', errors='ignore')
        
        # Reindex with all dates and fill missing with 0
        magnitude_summary_without_total.index = pd.to_datetime(magnitude_summary_without_total.index)
        magnitude_summary_without_total.index = magnitude_summary_without_total.index.strftime('%d-%m-%Y')

        # Plot setup
        fig, ax = plt.subplots(figsize=(10, 6))
        bar_width = 0.25
        x = np.arange(len(magnitude_summary_without_total.index))

        # Plot each category
        for i, category in enumerate(categories):
            ax.bar(x + i * bar_width, 
                magnitude_summary_without_total[category], 
                bar_width, 
                label=category, 
                color=colors[i], 
                edgecolor='black')

        # Chart formatting
        ax.set_title('Frekuensi Gempa Berdasarkan Magnitudo',fontsize=14, pad=20)
        ax.set_xlabel('Tanggal', fontsize=12)
        ax.set_ylabel('Jumlah Kejadian', fontsize=12)
        
        # Set x-ticks to show all dates
        ax.set_xticks(x + bar_width)
        
        # Update x-tick labels dengan format baru
        ax.set_xticklabels(magnitude_summary_without_total.index, rotation=45, ha='right', fontsize=8)
        
        ax.legend(title='Kategori Magnitudo', fontsize=10)
        ax.grid(axis='y', linestyle='--', alpha=0.7)

        # Add value labels
        for p in ax.patches:
            height = p.get_height()
            if height > 0:  # Hanya tampilkan label jika nilai > 0
                ax.annotate(f'{int(height)}', 
                        (p.get_x() + p.get_width() / 2, height),
                        ha='center', va='bottom', fontsize=8)

        plt.tight_layout()
        plt.savefig(os.path.join(nama_folder, 'diagbat_mag.png'), dpi=300)
        plt.close()

        # Insert to Excel
        worksheet_mag = writer.sheets['Magnitude Summary']
        worksheet_mag.insert_image('A10', os.path.join(nama_folder, 'diagbat_mag.png'), 
                                {'x_scale': 1.2, 'y_scale': 1.4})

        # Buat diagram lingkaran magnitude
        magnitude_total = magnitude_summary.loc['Jumlah Gempa', ['M<3', '3≤M<5', 'M≥5']]
        labels = ['M<3', '3≤M<5', 'M≥5']
        colors = ['green', 'yellow', 'red']

        # Membuat figure dan pie chart
        fig, ax = plt.subplots(figsize=(6, 6))

        # Fungsi untuk menampilkan persentase dan jumlah kejadian di setiap segmen
        def func(pct, allvals):
            absolute = int(np.round(pct/100.*np.sum(allvals)))
            if absolute > 0:
                return f'{pct:.1f}%\n({absolute} kejadian)'
            else:
                return ''  # Jika tidak ada kejadian, label tidak ditampilkan

        # Membuat pie chart
        wedges, texts, autotexts = ax.pie(
            magnitude_total,
            labels=labels,
            colors=colors,
            startangle=85,
            autopct=lambda pct: func(pct, magnitude_total),  # Menampilkan persentase dan jumlah kejadian
            pctdistance=0.875,  # Jarak persentase dari pusat pie
            labeldistance=9  # Jarak label dari pusat pie (aslinya 1.17)
        )

        # Mengatur ukuran font dan latar belakang putih untuk label dan persentase
        for text in texts:
            text.set_fontsize(12)
            text.set_bbox(dict(facecolor='white', edgecolor='none', boxstyle='round,pad=0.2'))

        for autotext in autotexts:
            autotext.set_fontsize(12)
            autotext.set_bbox(dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.2'))

        plt.title('Distribusi Magnitudo', fontsize=11)
        ax.legend(title='Kategori Magnitudo', fontsize=7, title_fontsize='9', loc='lower right')
        plt.tight_layout()

        plt.savefig(os.path.join(nama_folder, 'diaglingk_mag.png'))

        # Insert the magnitude pie chart into Excel
        worksheet_mag.insert_image('W10', os.path.join(nama_folder, 'diaglingk_mag.png'), {'x_scale': 1.2, 'y_scale': 1.2})
        
        # ================== DEPTH BAR CHART ==================
        categories_depth = ['D≤60 km', '60<D≤300 km', 'D>300 km']
        colors_depth = ['red', 'yellow', 'green']
        
       # Remove total row
        depth_summary_without_total = depth_summary.drop(index='Jumlah Gempa', errors='ignore')
        
        # Reindex with all dates and fill missing with 0
        depth_summary_without_total.index = pd.to_datetime(depth_summary_without_total.index)
        depth_summary_without_total.index = depth_summary_without_total.index.strftime('%d-%m-%Y')

        # Plot setup
        fig, ax = plt.subplots(figsize=(10, 6))
        bar_width = 0.25
        x = np.arange(len(depth_summary_without_total.index))
        
        # Plot each category
        for i, category in enumerate(categories_depth):
            ax.bar(x + i * bar_width, 
                depth_summary_without_total[category], 
                bar_width, 
                label=category, 
                color=colors_depth[i], 
                edgecolor='black')

        # Chart formatting
        ax.set_title('Frekuensi Gempa Berdasarkan Kedalaman' ,fontsize=14, pad=20)
        ax.set_xlabel('Tanggal', fontsize=12)
        ax.set_ylabel('Jumlah Kejadian', fontsize=12)
        
        # Set x-ticks to show all dates
        ax.set_xticks(x + bar_width)
        
        # Update x-tick labels dengan format baru
        ax.set_xticklabels(depth_summary_without_total.index, rotation=45, ha='right', fontsize=8)
        
        ax.legend(title='Kategori Kedalaman', fontsize=10)
        ax.grid(axis='y', linestyle='--', alpha=0.7)

        for p in ax.patches:
            height = p.get_height()
            if height > 0:  # Hanya tampilkan label jika nilai > 0
                ax.annotate(f'{int(height)}', 
                        (p.get_x() + p.get_width() / 2, height),
                        ha='center', va='bottom', fontsize=8)

        plt.tight_layout()
        plt.savefig(os.path.join(nama_folder, 'diagbat_depth.png'), dpi=300)
        plt.close()

        worksheet_depth = writer.sheets['Depth Summary']
        worksheet_depth.insert_image('A10', os.path.join(nama_folder, 'diagbat_depth.png'), 
                                {'x_scale': 1.2, 'y_scale': 1.4})

        # buat digram lingkaran depth
        depth_total = depth_summary.loc['Jumlah Gempa', ['D≤60 km', '60<D≤300 km', 'D>300 km']]
        labels = ['D≤60 km', '60<D≤300 km', 'D>300 km']
        colors = ['red', 'yellow', 'green']
        
        # Membuat figure dan pie chart
        fig, ax = plt.subplots(figsize=(6, 6))
        
        # Fungsi untuk menampilkan persentase dan jumlah kejadian di setiap segmen
        def func(pct, allvals):
            absolute = int(np.round(pct/100.*np.sum(allvals)))
            if absolute > 0:
                return f'{pct:.1f}%\n({absolute} kejadian)'
            else:
                return ''  # Jika tidak ada kejadian, label tidak ditampilkan
                  
        # Membuat pie chart
        wedges, texts, autotexts = ax.pie(
            depth_total,
            labels=labels,
            colors=colors,
            startangle=85,
            autopct=lambda pct: func(pct, magnitude_total),  # Menampilkan persentase dan jumlah kejadian
            pctdistance=0.975,  # Jarak persentase dari pusat pie
            labeldistance=9  # Jarak label dari pusat pie (aslinya 1.15)
        )

        # Mengatur ukuran font dan latar belakang putih untuk label dan persentase
        for text in texts:
            text.set_fontsize(12)
            text.set_bbox(dict(facecolor='white', edgecolor='none', boxstyle='round,pad=0.2'))

        for autotext in autotexts:
            autotext.set_fontsize(12)
            autotext.set_bbox(dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.2'))

        plt.title('Distribusi Kedalaman', fontsize=11)
        ax.legend(title='Kategori Kedalaman', fontsize=7, title_fontsize='9', loc='lower right')
        plt.tight_layout()
        plt.savefig(os.path.join(nama_folder, 'diaglingk_depth.png'))

        # Insert the magnitude pie chart into Excel
        worksheet_depth.insert_image('W10', os.path.join(nama_folder, 'diaglingk_depth.png'), {'x_scale': 1.2, 'y_scale': 1.2})   
    
    def autofit_columns(file_path, sheet_name):
        """
        Menyesuaikan lebar kolom di Excel berdasarkan panjang konten.
        """
        workbook = load_workbook(file_path)  # Membuka workbook yang telah disimpan
        worksheet = workbook[sheet_name]  # Mendapatkan worksheet berdasarkan nama sheet

        for col in worksheet.columns:
            max_length = 0
            column = col[0].column_letter  # Mendapatkan huruf kolom, misalnya 'A', 'B', dst.
            for cell in col:
                try:
                    # Mencari panjang maksimum dari setiap kolom
                    if len(str(cell.value)) > max_length:
                        max_length = len(str(cell.value))
                except:
                    pass
            adjusted_width = (max_length + 2)  # Menambah padding
            worksheet.column_dimensions[column].width = adjusted_width  # Mengatur lebar kolom

        workbook.save(file_path)  # Menyimpan workbook setelah penyesuaian
        workbook.close()  # Menutup workbook untuk memastikan semua perubahan disimpan

    # Ambil tanggal dari input pengguna
    # start = start_date_entry.get()
    # end = end_date_entry.get()
    
    # Panggil fungsi utama dengan tanggal yang dipilih
    read_csv_and_analyze(tanggal_awal, tanggal_akhir)

def print_map():
   #membuat file .dat
    df = pd.read_csv('datalengkap.csv')

    # Memilih kolom yang diinginkan
    selected_columns = df[['Bujur','Lintang','Kedalaman', 'Magnitude']]

    # Menulis data ke file DAT
    with open('inputpeta.dat', 'w') as file:
        for index, row in selected_columns.iterrows():
            line = '  '.join(row.astype(str))  # Menggabungkan semua elemen baris menjadi satu string dengan spasi sebagai pemisah
            file.write(line + '\n')
            
    # Membaca data dari hasil2.csv
    data = pd.read_csv('datalengkap.csv')

    # Buat figure
    fig = pygmt.Figure()

    # Buat color palette untuk kedalaman gempa
    pygmt.makecpt(cmap="geo", series=[-7000, 4000])
    pygmt.makecpt(cmap="red,yellow,green", series="0,60,300,1000", output="quakes.cpt")
    start_date_formatted = start_date_entry.get().replace('/', '-')  # Ganti format tanggal
    end_date_formatted = end_date_entry.get().replace('/', '-')
    #title_text = f"Peta Seismisitas Wilayah Bali dan Sekitarnya \nPeriode {start_date_formatted} hingga {end_date_formatted}"
    data['Magnitude'] = pd.to_numeric(data['Magnitude'], errors='coerce')  # Ganti nilai tidak valid dengan NaN

    # Plot peta dasar dengan topografi
    fig.grdimage(
        grid="D:/Seismioto/Bahan/earth_relief_15s.grd",
        region=[113.21, 117.31, -12, -7],
        projection="M17c",
        shading=True,
        frame=["xa2g2", "ya2g2"])#'+t"Peta Seismisitas"'])
    
    # Plot fault lines
    fig.plot(data="D:/Seismioto/Bahan/fault.gmt", pen="0.6,black")
    fig.plot(data="D:/Seismioto/Bahan/trench.gmt", pen="4,black")

    fig.plot(x=[114.8, 115.4], y=[-11.5, -7.5], projection="M", pen=2)
    fig.text(x=114.9, y=-11.6, text="A", font="18,Helvetica")
    fig.text(x=115.55, y=-7.5, text="A1", font="18,Helvetica")

    # Plot gempa yang tidak dirasakan (kolom 'Dirasakan' kosong)
    gempa_tidak_dirasakan = data[data['Dirasakan'].isna()]
    if not gempa_tidak_dirasakan.empty:
        fig.plot(
            x=gempa_tidak_dirasakan['Bujur'],
            y=gempa_tidak_dirasakan['Lintang'],
            size=0.08*gempa_tidak_dirasakan['Magnitude'],
            style="cc",  # Lingkaran
            fill=gempa_tidak_dirasakan['Kedalaman'].astype(float),  # Warna berdasarkan kedalaman
            cmap="quakes.cpt",  # Gunakan colormap yang telah ditentukan
            pen="black"
        )
    else:
        print("ada gempa bumi yang dirasakan untuk diplot.")

    # Plot gempa yang dirasakan (kolom 'Dirasakan' terisi)
    gempa_dirasakan = data[~data['Dirasakan'].isna()]
    if not gempa_dirasakan.empty:
        fig.plot(
            x=gempa_dirasakan['Bujur'],
            y=gempa_dirasakan['Lintang'],
            size=0.08*gempa_dirasakan['Magnitude'],
            style="a0.5c",  # Simbol bintang
            fill="red",  # Warna bintang merah
            pen="1p,black"  # Garis tepi hitam dengan ketebalan 1 point
        )
    else:
        print("tidak ada gempa bumi yang dirasakan untuk diplot.")

       
    # Menambahkan colorbar
    fig.image('D:/Seismioto/Bahan/LogoBMKG.png', position='n1.1/0.85+w1.1i')
    fig.image('D:/Seismioto/Bahan/mtangn.png', position='n1.04/0.7+w1.1i')
    fig.basemap(map_scale="n1.12/0.65+w80k+f0.1p+lKm")
    fig.colorbar(
        frame=["x+lKetinggian", "y"],  # Label untuk sumbu x dan y
        position="n1.24/0.63+w4c/0.5c")  # Posisi colorbar (JMR = tengah-kanan), lebar 0.5 cm, panjang 10 cm
    fig.legend(spec="D:/Seismioto/Bahan/legenda.gmt", position="n1.03/0.04+w5c",box="+gwhite+p1p,black")
    with fig.inset(position="n1.03/0.28+w5c/2.5c", box="+pblack"):
        fig.coast(
            region=[97, 140, -15, 7],
            shorelines='0.5p,black',
            projection="M5c",
            land="green",
            water="lightblue"
        )
        fig.plot(x=[117.31, 113.21, 113.21, 117.31, 117.31], y=[-7, -7, -12, -12, -7], pen="1p,red")

    # Membuat cross section
    pygmt.project(
        data="inputpeta.dat",
        unit=True,
        center=[114.8, -11.5],
        endpoint=[115.4, -7.5],
        width=[-200, 200],
        convention="pz",
        outfile="crsx.dat"
    )

    fig.shift_origin(yshift=9, xshift=17.5)
    fig.plot(
        x=[113, 113, 114.18, 114.18],  # Koordinat X dari sudut-sudut persegi
        y=[-11.4, -12, -12, -11.4],  # Koordinat Y dari sudut-sudut persegi
        fill="white",  # Warna latar belakang
        close=True  # Menutup jalur untuk membuat persegi berisi warna 
    )

    # Buat peta cross section dengan basemap
    fig.basemap(
        projection="X4c/-2.5c",
        region=[0, 450, 0, 350],
        frame=['xafg+lDistance (km)','yafg+lDepth (km)', "wsEN"]
    )

     # Membuat garis cross section   
    cross = pd.read_csv('crsx.dat', delimiter='\s+', header=None)
    # Define your column names
    header = ['Distance', 'Dept', 'Mag']
    
    # Assign the header
    cross.columns = header

    fig.plot(x=cross.Distance, 
              y=cross.Dept, 
              projection="X4/-2.5", 
              size =0.035 * (2 * cross.Mag), style="cc", 
              fill = cross.Dept, 
              cmap = "quakes.cpt", 
              pen = 'black'),
              #region = region_cs)
    fig.text(x=20, y=25, text="A", font="11,Helvetica")
    fig.text(x=370, y=25, text="A1", font="11,Helvetica")

    # Tentukan nama folder berdasarkan tanggal awal dan akhir
    if tanggal_awal.month == tanggal_akhir.month:
        nama_folder = f"{tanggal_awal.day:02d} - {tanggal_akhir.day:02d} {tanggal_awal.strftime('%B %Y')}"
    else:
        nama_folder = f"{tanggal_awal.day:02d} {tanggal_awal.strftime('%B')} - {tanggal_akhir.day:02d} {tanggal_akhir.strftime('%B %Y')}"

    output_map_filename = os.path.join(nama_folder, f'Peta Seismisitas Periode {nama_folder}.png')
    # Menyimpan gambar
    fig.savefig(fname=output_map_filename)
    #fig.show()
    
    messagebox.showinfo("Cetak Peta", "Peta Berhasil di Cetak")

def print_report():
    # Set locale to Indonesian for date formatting
    locale.setlocale(locale.LC_TIME, 'id_ID.UTF-8')

    # Fungsi untuk memilih file
    def select_file():
        root = tk.Tk()
        root.withdraw()  # Menyembunyikan jendela utama
        file_path = filedialog.askopenfilename(title="Pilih file laporan_gempa.xlsx", filetypes=[("Excel files", "*.xlsx")])
        return file_path

    # Memilih file Excel
    file_path = select_file()
    if not file_path:
        print("Tidak ada file yang dipilih.")
    else:
        # Load the Excel file
        earthquake_data = pd.read_excel(file_path)

        # Pastikan kolom 'Tanggal' dalam format datetime
        earthquake_data['Tanggal'] = pd.to_datetime(earthquake_data['Tanggal'])
        
        # Tentukan nama folder berdasarkan tanggal awal dan akhir
        if tanggal_awal.month == tanggal_akhir.month:
            nama_folder = f"{tanggal_awal.day:02d} - {tanggal_akhir.day:02d} {tanggal_awal.strftime('%B %Y')}"
        else:
            nama_folder = f"{tanggal_awal.day:02d} {tanggal_awal.strftime('%B')} - {tanggal_akhir.day:02d} {tanggal_akhir.strftime('%B %Y')}"

        # Ambil tanggal awal dan akhir
        start_date = tanggal_awal
        end_date = tanggal_akhir
  
        # Path to images
        logo_path = "D:/Seismioto/Bahan/LogoBMKG.png"  # Ganti dengan path logo BMKG
        magnitude_bar_chart_path = f"D:/Seismioto/Pengolahan/{nama_folder}/diagbat_mag.png"
        magnitude_pie_chart_path = f"D:/Seismioto/Pengolahan/{nama_folder}/diaglingk_mag.png"
        depth_bar_chart_path = f"D:/Seismioto/Pengolahan/{nama_folder}/diagbat_depth.png"
        depth_pie_chart_path = f"D:/Seismioto/Pengolahan/{nama_folder}/diaglingk_depth.png"
        seismic_map_path = f"D:/Seismioto/Pengolahan/{nama_folder}/Peta Seismisitas Periode {nama_folder}.png"
       
        earthquake_data = pd.read_excel(file_path)

        # Pastikan kolom 'Tanggal' dalam format datetime
        earthquake_data['Tanggal'] = pd.to_datetime(earthquake_data['Tanggal'])

        # Calculate the required values
        total_earthquakes = len(earthquake_data)
        min_magnitude = earthquake_data['Magnitude'].min()
        max_magnitude = earthquake_data['Magnitude'].max()

        # Format tanggal dengan nama bulan dalam bahasa Indonesia
        start_date_formatted = start_date.strftime('%d %B %Y')  # contoh format: 01 Januari 2024
        end_date_formatted = end_date.strftime('%d %B %Y')      # contoh format: 05 September 2024
        #date_formatted=date1.strftime('%d %B %Y')
        #print(date_formatted)

        # Menentukan tanggal awal bulan untuk memulai perhitungan minggu
        start_of_month = start_date.replace(day=1)

        # Hitung nomor minggu relatif terhadap awal bulan
        week_number = (start_date - start_of_month).days // 7 + 1

        # Format bulan dalam bahasa Indonesia
        bulan_formatted = start_date.strftime('%B')

        # Menghitung jumlah kejadian gempabumi total
        total_earthquakes = len(earthquake_data)

        # Menghitung jumlah gempa dengan lat > -9 dan kedalaman < 60 km
        shallow_earthquake_count = earthquake_data[(earthquake_data['Lintang'] > -9) & 
                                                   (earthquake_data['Kedalaman'] < 60)].shape[0]

        # Menghitung jumlah gempa di selatan Pulau Jawa, Bali, dan Lombok
        southern_earthquake_count = total_earthquakes - shallow_earthquake_count

        # Magnitude categories
        magnitude_below_3 = earthquake_data[earthquake_data['Magnitude'] <= 3]
        magnitude_3_to_5 = earthquake_data[(earthquake_data['Magnitude'] > 3) & (earthquake_data['Magnitude'] <= 5)]
        magnitude_above_5 = earthquake_data[earthquake_data['Magnitude'] > 5]

        count_below_3 = len(magnitude_below_3)
        count_3_to_5 = len(magnitude_3_to_5)
        count_above_5 = len(magnitude_above_5)

        percentage_below_3 = int((count_below_3 / total_earthquakes) * 100)
        percentage_3_to_5 = int((count_3_to_5 / total_earthquakes) * 100)
        percentage_above_5 = int((count_above_5 / total_earthquakes) * 100)

        # Magnitude categories
        depth_below_60 = earthquake_data[earthquake_data['Kedalaman'] < 60]
        depth_60_to_300 = earthquake_data[(earthquake_data['Kedalaman'] >= 60) & (earthquake_data['Kedalaman'] <= 300)]
        depth_above_300 = earthquake_data[earthquake_data['Kedalaman'] > 300]

        count_depth_below_60 = len(depth_below_60)
        count_depth_60_to_300 = len(depth_60_to_300)
        count_depth_above_300 = len(depth_above_300)

        percentage_depth_below_60 = int((count_depth_below_60 / total_earthquakes) * 100)
        percentage_depth_60_to_300 = int((count_depth_60_to_300 / total_earthquakes) * 100)
        percentage_depth_above_300 = int((count_depth_above_300 / total_earthquakes) * 100)

        # Determine dominant depth category
        dominant_category = ''
        if percentage_depth_below_60 >= percentage_depth_60_to_300 and percentage_depth_below_60 >= percentage_depth_above_300:
            dominant_category = f'dangkal (<60 km) sebesar {percentage_depth_below_60}%'
        elif percentage_depth_60_to_300 >= percentage_depth_below_60 and percentage_depth_60_to_300 >= percentage_depth_above_300:
            dominant_category = f'kedalaman menengah (60 - 300 km) sebesar {percentage_depth_60_to_300}%'
        else:
            dominant_category = f'kedalaman dalam (>300 km) sebesar {percentage_depth_above_300}%'

        # Create a new Word document
        document = Document()

        # Ukuran kertas F4 (21.0 cm x 33.0 cm)
        f4_width = Cm(21.0)
        f4_height = Cm(33.0)

        # Mengatur ukuran kertas F4 untuk setiap section di dokumen
        for section in document.sections:
            # Mengatur orientasi potrait untuk F4
            section.orientation = WD_ORIENT.PORTRAIT  # Bisa diubah ke WD_ORIENT.LANDSCAPE jika ingin landscape
            section.page_width = f4_width
            section.page_height = f4_height

            # Mengatur margin halaman (optional, bisa disesuaikan)
            section.top_margin = Cm(2.5)     # Marginn atas
            section.bottom_margin = Cm(2.5)  # Marginn bawah
            section.left_margin = Cm(2.5)    # Marginn kiri
            section.right_margin = Cm(2.5)   # Marginn kanan

        # Add Logo at the top
        # logo_paragraph = document.add_paragraph()
        # logo_run = logo_paragraph.add_run()
        # logo_run.add_picture(logo_path, width=Inches(1))  # Menambahkan logo dengan lebar 1.5 inches
        # logo_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

        # # Add Title
        title = document.add_paragraph()
        title_run = title.add_run(f'STASIUN GEOFISIKA DENPASAR                                                                                              LAPORAN AKTIVITAS SEISMISITAS WILAYAH BALI DAN SEKITARNYA                                                                                              PERIODE {nama_folder}')
        title_run.bold = True
        title_run.font.size = Pt(14)
        title_run.font.all_caps = True  # Membuat semua huruf kapital
        title.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

        # Add Subtitle
        # subtitle = document.add_paragraph()
        # subtitle_run = subtitle.add_run(f'STASIUN GEOFISIKA DENPASAR                                                                                              LAPORAN AKTIVITAS SEISMISITAS WILAYAH BALI DAN SEKITARNYA                                                                                              PERIODE {start_date_formatted} – {end_date_formatted}')
        # subtitle_run.font.size = Pt(12)
        # subtitle_run.font.all_caps = True  # Membuat semua huruf kapital
        # subtitle.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

        # # Add Date Period
        # period = document.add_paragraph()
        # period_run = period.add_run(f'PERIODE {start_date_formatted} – {end_date_formatted}')
        # period_run.font.size = Pt(12)
        # period_run.font.all_caps = True  # Membuat semua huruf kapital
        # period.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

        # Add space
        document.add_paragraph('')

        # Teks Deskriptif Magnitudo dengan perhitungan minggu bulanan
        magnitude_descriptive_text = (
            f"Berdasarkan data Stasiun Geofisika Denpasar selama minggu ke-{week_number} bulan "
            f"{bulan_formatted} {start_date.year}, "
            f"di daerah Bali dan sekitarnya telah terjadi {total_earthquakes} kejadian gempabumi dengan magnitudo bervariasi "
            f"mulai dari M {min_magnitude} sampai M {max_magnitude}. Kejadian gempabumi dengan magnitudo M<3 sejumlah "
            f"{count_below_3} kejadian atau {percentage_below_3}% dari total kejadian gempabumi. Sedangkan untuk M 3 – 5 "
            f"sejumlah {count_3_to_5} kejadian atau {percentage_3_to_5}% dari total kejadian gempabumi "
        )

        # Conditional text for magnitude above 5
        if count_above_5 > 0:
            magnitude_descriptive_text += (
                f"dan kejadian gempa M>5 sejumlah {count_above_5} kejadian atau {percentage_above_5}% dari total kejadian gempabumi."
            )
        else:
            magnitude_descriptive_text += "dan tidak terdapat kejadian gempabumi dengan magnitudo M>5."

        # Menambahkan paragraf dengan teks deskriptif magnitudo
        magnitude_paragraph = document.add_paragraph(magnitude_descriptive_text)

        # Mengatur alignment paragraf menjadi justify
        magnitude_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY

        # Add Magnitude Bar Chart Image
        try:
            document.add_paragraph().add_run().add_picture(magnitude_bar_chart_path, width=Inches(5.65))
            document.paragraphs[-1].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
        except FileNotFoundError:
            document.add_paragraph('DIAGRAM BATANG MAGNITUDO TIDAK DITEMUKAN').bold = True

        # Add Magnitude Pie Chart Image
        try:
            document.add_paragraph().add_run().add_picture(magnitude_pie_chart_path, width=Inches(3.15))
            document.paragraphs[-1].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
        except FileNotFoundError:
            document.add_paragraph('DIAGRAM LINGKARAN MAGNITUDO TIDAK DITEMUKAN').bold = True

        # Teks Keterangan Gambar Magnitudo
        document.add_paragraph('Gambar 1. Histogram gempabumi harian berdasarkan magnitudo (atas) dan diagram gempabumi berdasarkan magnitudo (bawah)').alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

        # Determine dominant depth category
        dominant_category = ''
        if percentage_depth_below_60 >= percentage_depth_60_to_300 and percentage_depth_below_60 >= percentage_depth_above_300:
            dominant_category = f'dangkal (<60 km) sebesar {percentage_depth_below_60}%'
        elif percentage_depth_60_to_300 >= percentage_depth_below_60 and percentage_depth_60_to_300 >= percentage_depth_above_300:
            dominant_category = f'kedalaman menengah (60 - 300 km) sebesar {percentage_depth_60_to_300}%'
        else:
            dominant_category = f'kedalaman dalam (>300 km) sebesar {percentage_depth_above_300}%'

        # Add space
        document.add_paragraph('')

        # Teks Deskriptif Kedalaman
        depth_descriptive_text = (
            f"Berdasarkan kedalaman, gempabumi yang mendominasi adalah kejadian gempabumi {dominant_category}  dari total kejadian gempabumi. "
            f"Jumlah gempabumi dangkal sebanyak {count_depth_below_60} kejadian gempabumi, "
            f"terdapat sebanyak {count_depth_60_to_300} kejadian gempabumi dengan kedalaman menengah 60 km - 300 km atau {percentage_depth_60_to_300}% dari total kejadian gempabumi, "
        )

        # Conditional text for depth above 300 km
        if count_depth_above_300 > 0:
            depth_descriptive_text += (
                f"dan kejadian gempa dalam sejumlah {count_depth_above_300} kejadian atau {percentage_depth_above_300}% dari total kejadian gempabumi."
            )
        else:
            depth_descriptive_text += "dan tidak terdapat kejadian gempabumi dengan kedalaman dalam >300 km."

        depth_paragraph = document.add_paragraph(depth_descriptive_text)
        depth_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY

        # Add Depth Bar Chart Image
        try:
            document.add_paragraph().add_run().add_picture(depth_bar_chart_path, width=Inches(5.65))
            document.paragraphs[-1].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
        except FileNotFoundError:
            document.add_paragraph('DIAGRAM BATANG KEDALAMAN TIDAK DITEMUKAN').bold = True

        # Add Depth Pie Chart Image
        try:
            document.add_paragraph().add_run().add_picture(depth_pie_chart_path, width=Inches(3.15))
            document.paragraphs[-1].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
        except FileNotFoundError:
            document.add_paragraph('DIAGRAM LINGKARAN KEDALAMAN TIDAK DITEMUKAN').bold = True

        # Teks Keterangan Gambar Kedalaman (akan diisi sesuai format)
        document.add_paragraph('Gambar 2.  Histogram gempabumi harian berdasarkan kedalaman (atas) dan diagram gempabumi berdasarkan kedalaman (bawah)').alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

        # Add Seismic Map Image
        try:
            document.add_paragraph().add_run().add_picture(seismic_map_path, width=Inches(6))
            document.paragraphs[-1].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
        except FileNotFoundError:
            document.add_paragraph('PETA SEISMISITAS TIDAK DITEMUKAN').bold = True

        # Teks Keterangan Gambar Peta Seismisitas
        peta_seismisitas_text = (
            f"Gambar 3. Peta seismisitas Bali dan sekitarnya periode {nama_folder}"
        )
        peta_seismisitas_paragraph = document.add_paragraph(peta_seismisitas_text)
        peta_seismisitas_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

        # Add space
        document.add_paragraph('')

        # Teks analisis yang diminta
        analisis_text1 = (
            f"Selama satu minggu terakhir, gempabumi yang terjadi di Bali dan sekitarnya dikelompokan berdasarkan sumbernya (Gambar 3):\n"
        )    

        # Menambahkan paragraf analisis ke dalam dokumen
        analisis_paragraph1 = document.add_paragraph(analisis_text1)
        analisis_paragraph1.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT

        analisis_text2 = (
            f"1. Sebanyak {southern_earthquake_count} kejadian gempabumi terjadi di selatan Pulau Jawa, Bali, dan Lombok "
            f"yang diperkirakan berasosiasi dengan subduksi Indo Australia-Eurasia.\n"
            f"2. Sebanyak {shallow_earthquake_count} kejadian gempabumi terjadi menyebar di wilayah Pulau Bali, Lombok dan sekitarnya "
            f"diperkirakan berkaitan dengan sesar di belakang busur Flores atau Flores Back Arc Thrust dan aktivitas sesar aktif."
        )

        # Menambahkan paragraf analisis ke dalam dokumen
        analisis_paragraph2 = document.add_paragraph(analisis_text2)
        analisis_paragraph2.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT

        # Menghitung jumlah kejadian gempa yang dirasakan
        earthquakes_felt = earthquake_data[earthquake_data['Dirasakan'].notna()]
        total_felt = len(earthquakes_felt)

        # Menyusun narasi pembuka
        if total_felt > 0:
            felt_intro = (
                f"Selama periode ini terdapat {total_felt} kejadian gempabumi yang dirasakan oleh masyarakat "
                f"di wilayah Bali dan sekitarnya."
            )
        else:
            felt_intro = (
                f"Selama periode ini tidak terdapat kejadian gempabumi yang dirasakan oleh masyarakat "
                f"di wilayah Bali dan sekitarnya."
            )

        # Menambahkan narasi pembuka ke dokumen
        felt_intro_paragraph = document.add_paragraph(felt_intro)
        felt_intro_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY

        # Menyusun detail setiap kejadian gempa yang dirasakan dalam format list dengan nomor urut
        if total_felt > 0:
            # Inisialisasi counter untuk nomor urut
            counter = 1

            for index, row in earthquakes_felt.iterrows():
                magnitude = row['Magnitude']
                date = row['Tanggal'].strftime('%d %B %Y')  # Format tanggal DD/MM/YYYY
                time = row['Waktu (WIB)']  # Asumsikan kolom waktu sudah dalam format string yang benar
                location = row['Dirasakan']
                description = row['Keterangan']
                depth = row['Kedalaman']

                # Format detail gempa yang dirasakan dengan nomor urut
                felt_detail = (
                    f"{counter}. Gempabumi dengan magnitudo {magnitude} {location} yang terjadi pada tanggal {date} "
                    f"pukul {time} WIB berlokasi di {description} pada kedalaman {depth} km."
                )

                # Menambahkan detail gempa ke dalam dokumen sebagai paragraf terpisah
                felt_detail_paragraph = document.add_paragraph(felt_detail)
                felt_detail_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY

                # Meningkatkan counter untuk nomor urut berikutnya
                counter += 1
        else:
            # Jika tidak ada gempa yang dirasakan, tambahkan keterangan tambahan
            no_felt_detail = document.add_paragraph(" ")
            no_felt_detail.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER


        # Tambahkan section baru dengan orientasi landscape
        new_section = document.add_section(WD_SECTION.NEW_PAGE)
        new_section.orientation = WD_ORIENT.LANDSCAPE

        # Atur ukuran kertas agar landscape sesuai
        new_section.page_width, new_section.page_height = new_section.page_height, new_section.page_width

        # Menentukan margin untuk section baru agar tabel berada di tengah secara vertikal
        section_margin = Cm(2)  # Sesuaikan margin jika perlu
        new_section.top_margin = section_margin
        new_section.bottom_margin = section_margin
        new_section.left_margin = section_margin
        new_section.right_margin = section_margin

        # Menambahkan paragraf baru untuk membungkus tabel
        table_paragraph = document.add_paragraph()
        table_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER  # Mengatur alignment paragraf ke tengah

        # Membuat teks keterangan tabel gempabumi
        tabel_keterangan_text = (
            f"Tabel 1. Data gempabumi di Bali dan sekitarnya tanggal {nama_folder}"
        )

        # Menambahkan teks keterangan tabel ke dalam dokumen
        tabel_keterangan_paragraph = document.add_paragraph(tabel_keterangan_text)
        tabel_keterangan_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER  # Mengatur alignment menjadi tengah

        # Menambahkan tabel langsung ke dalam paragraf
        table = document.add_table(rows=1, cols=len(earthquake_data.columns))
        table.style = 'Table Grid'

        # Set table alignment to center
        table.alignment = WD_TABLE_ALIGNMENT.CENTER

        # Set column widths to match the text content
        column_widths = {
            "NO.": 1.5,  # NO
            "Tanggal": 2.5,  # TANGGAL
            "Waktu (WIB)": 2.5,  # WAKTU (WIB)
            "Lintang": 2.0,  # LATITUDE
            "Bujur": 2.0,  # LONGITUDE
            "Kedalaman": 2.0,  # Depth
            "Magnitude": 1.5,  # Magnitude
            "Keterangan": 5.0,  # Keterangan
            "Dirasakan": 2.5  # Dirasakan
        }

        # Add header row
        hdr_cells = table.rows[0].cells
        for i, column_name in enumerate(earthquake_data.columns):
            hdr_cells[i].text = column_name
            hdr_cells[i].width = Cm(column_widths.get(column_name, 2.0))  # Set width based on column

        # Add data rows, memformat kolom 'Tanggal' tanpa jam
        for index, row in earthquake_data.iterrows():
            row_cells = table.add_row().cells
            for i, value in enumerate(row):
                # Jika kolom adalah 'Tanggal', hanya tampilkan tanggal tanpa jam
                if earthquake_data.columns[i] == 'Tanggal':
                    row_cells[i].text = value.strftime('%d/%m/%Y')  # Format tanggal menjadi DD/MM/YYYY
                # Jika kolom adalah 'Dirasakan' dan nilai adalah NaN, ganti dengan teks yang sesuai
                elif earthquake_data.columns[i] == 'Dirasakan':
                    # Ganti NaN dengan string kosong atau teks lain
                    row_cells[i].text = '' if pd.isna(value) else str(value)
                else:
                    row_cells[i].text = str(value)

                # Set the width of each data cell, sesuai dengan kolom
                row_cells[i].width = Cm(column_widths.get(earthquake_data.columns[i], 2.0))

        # Mengatur tabel agar berada di tengah secara horizontal
        table.alignment = WD_TABLE_ALIGNMENT.CENTER
        
        
        # Simpan 'datalengkap.csv' ke folder baru dengan nama 'data gempa (nama folder).xlsx'
        output_file = os.path.join(nama_folder, f'Laporan Gempabumi Periode {nama_folder}.docx')
    
        # Save the document
        document.save(output_file)

    messagebox.showinfo("Cetak Laporan", "Berhasil Mencetak laporan ke file Word")

def print_pressrelease():
    # Set locale to Indonesian for date formatting
    locale.setlocale(locale.LC_TIME, 'id_ID.UTF-8')

    # Fungsi untuk memilih file
    def select_file():
        root = tk.Tk()
        root.withdraw()  # Menyembunyikan jendela utama
        file_path = filedialog.askopenfilename(title="Pilih file laporan_gempa.xlsx", filetypes=[("Excel files", "*.xlsx")])
        return file_path

    # Memilih file Excel
    file_path = select_file()
    if not file_path:
        print("Tidak ada file yang dipilih.")
    else:
        # Load the Excel file
        earthquake_data = pd.read_excel(file_path)

        # Pastikan kolom 'Tanggal' dalam format datetime
        earthquake_data['Tanggal'] = pd.to_datetime(earthquake_data['Tanggal'])

        # Tentukan nama folder berdasarkan tanggal awal dan akhir
        if tanggal_awal.month == tanggal_akhir.month:
            nama_folder = f"{tanggal_awal.day:02d} - {tanggal_akhir.day:02d} {tanggal_awal.strftime('%B %Y')}"
        else:
            nama_folder = f"{tanggal_awal.day:02d} {tanggal_awal.strftime('%B')} - {tanggal_akhir.day:02d} {tanggal_akhir.strftime('%B %Y')}"

        # Ambil tanggal awal dan akhir
        start_date = tanggal_awal
        end_date = tanggal_akhir
  
        # Path to images
        magnitude_bar_chart_path = f"D:/Seismioto/Pengolahan/{nama_folder}/diagbat_mag.png"
        magnitude_pie_chart_path = f"D:/Seismioto/Pengolahan/{nama_folder}/diaglingk_mag.png"
        depth_bar_chart_path = f"D:/Seismioto/Pengolahan/{nama_folder}/diagbat_depth.png"
        depth_pie_chart_path = f"D:/Seismioto/Pengolahan/{nama_folder}/diaglingk_depth.png"
        seismic_map_path = f"D:/Seismioto/Pengolahan/{nama_folder}/Peta Seismisitas Periode {nama_folder}.png"
       
        earthquake_data = pd.read_excel(file_path)

        # Pastikan kolom 'Tanggal' dalam format datetime
        earthquake_data['Tanggal'] = pd.to_datetime(earthquake_data['Tanggal'])

        # Calculate the required values
        total_earthquakes = len(earthquake_data)
        min_magnitude = earthquake_data['Magnitude'].min()
        max_magnitude = earthquake_data['Magnitude'].max()

        # Format tanggal dengan nama bulan dalam bahasa Indonesia
        start_date_formatted = start_date.strftime('%d %B %Y')  # contoh format: 01 Januari 2024
        end_date_formatted = end_date.strftime('%d %B %Y')      # contoh format: 05 September 2024

        # Menentukan tanggal awal bulan untuk memulai perhitungan minggu
        start_of_month = start_date.replace(day=1)

        # Hitung nomor minggu relatif terhadap awal bulan
        week_number = (start_date - start_of_month).days // 7 + 1

        # Format bulan dalam bahasa Indonesia
        bulan_formatted = start_date.strftime('%B')

        # Menghitung jumlah kejadian gempabumi total
        total_earthquakes = len(earthquake_data)

        # Menghitung jumlah gempa dengan lat > -9 dan kedalaman < 60 km
        shallow_earthquake_count = earthquake_data[(earthquake_data['Lintang'] > -9) & 
                                                   (earthquake_data['Kedalaman'] < 60)].shape[0]

        # Menghitung jumlah gempa di selatan Pulau Jawa, Bali, dan Lombok
        southern_earthquake_count = total_earthquakes - shallow_earthquake_count

        # Magnitude categories
        magnitude_below_3 = earthquake_data[earthquake_data['Magnitude'] <= 3]
        magnitude_3_to_5 = earthquake_data[(earthquake_data['Magnitude'] > 3) & (earthquake_data['Magnitude'] <= 5)]
        magnitude_above_5 = earthquake_data[earthquake_data['Magnitude'] > 5]

        count_below_3 = len(magnitude_below_3)
        count_3_to_5 = len(magnitude_3_to_5)
        count_above_5 = len(magnitude_above_5)

        percentage_below_3 = int((count_below_3 / total_earthquakes) * 100)
        percentage_3_to_5 = int((count_3_to_5 / total_earthquakes) * 100)
        percentage_above_5 = int((count_above_5 / total_earthquakes) * 100)

        # Magnitude categories
        depth_below_60 = earthquake_data[earthquake_data['Kedalaman'] < 60]
        depth_60_to_300 = earthquake_data[(earthquake_data['Kedalaman'] >= 60) & (earthquake_data['Kedalaman'] <= 300)]
        depth_above_300 = earthquake_data[earthquake_data['Kedalaman'] > 300]

        count_depth_below_60 = len(depth_below_60)
        count_depth_60_to_300 = len(depth_60_to_300)
        count_depth_above_300 = len(depth_above_300)

        percentage_depth_below_60 = int((count_depth_below_60 / total_earthquakes) * 100)
        percentage_depth_60_to_300 = int((count_depth_60_to_300 / total_earthquakes) * 100)
        percentage_depth_above_300 = int((count_depth_above_300 / total_earthquakes) * 100)

        # Determine dominant depth category
        dominant_category = ''
        if percentage_depth_below_60 >= percentage_depth_60_to_300 and percentage_depth_below_60 >= percentage_depth_above_300:
            dominant_category = f'dangkal (<60 km) sebesar {percentage_depth_below_60}%'
        elif percentage_depth_60_to_300 >= percentage_depth_below_60 and percentage_depth_60_to_300 >= percentage_depth_above_300:
            dominant_category = f'kedalaman menengah (60 - 300 km) sebesar {percentage_depth_60_to_300}%'
        else:
            dominant_category = f'kedalaman dalam (>300 km) sebesar {percentage_depth_above_300}%'

        # Create a new Word document
        document = Document()

        # Ukuran kertas F4 (21.0 cm x 33.0 cm)
        f4_width = Cm(21.0)
        f4_height = Cm(33.0)

        # Mengatur ukuran kertas F4 untuk setiap section di dokumen
        for section in document.sections:
            # Mengatur orientasi potrait untuk F4
            section.orientation = WD_ORIENT.PORTRAIT  # Bisa diubah ke WD_ORIENT.LANDSCAPE jika ingin landscape
            section.page_width = f4_width
            section.page_height = f4_height

            # Mengatur margin halaman (optional, bisa disesuaikan)
            section.top_margin = Cm(2.5)     # Marginn atas
            section.bottom_margin = Cm(2.5)  # Marginn bawah
            section.left_margin = Cm(2.5)    # Marginn kiri
            section.right_margin = Cm(2.5)   # Marginn kanan

        # Add Title
        title = document.add_paragraph()
        title_run = title.add_run(f'PRESS RELEASE AKTIVITAS SEISMISITAS                                                                                              WILAYAH BALI DAN SEKITARNYA                                                                                                        PERIODE {nama_folder}')
        title_run.bold = True
        title_run.font.size = Pt(14)
        title_run.font.all_caps = True  # Membuat semua huruf kapital
        title.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

        # Add space
        document.add_paragraph('')

        # Teks Deskriptif Magnitudo dengan perhitungan minggu bulanan
        magnitude_descriptive_text = (
            f"Berdasarkan data Stasiun Geofisika Denpasar selama minggu ke-{week_number} bulan "
            f"{bulan_formatted} {start_date.year}, "
            f"di daerah Bali dan sekitarnya telah terjadi {total_earthquakes} kejadian gempabumi dengan magnitudo bervariasi "
            f"mulai dari M {min_magnitude} sampai M {max_magnitude}. Kejadian gempabumi dengan magnitudo M<3 sejumlah "
            f"{count_below_3} kejadian atau {percentage_below_3}% dari total kejadian gempabumi. Sedangkan untuk M 3 – 5 "
            f"sejumlah {count_3_to_5} kejadian atau {percentage_3_to_5}% dari total kejadian gempabumi "
        )

        # Conditional text for magnitude above 5
        if count_above_5 > 0:
            magnitude_descriptive_text += (
                f"dan kejadian gempa M>5 sejumlah {count_above_5} kejadian atau {percentage_above_5}% dari total kejadian gempabumi."
            )
        else:
            magnitude_descriptive_text += "dan tidak terdapat kejadian gempabumi dengan magnitudo M>5."

        # Menambahkan paragraf dengan teks deskriptif magnitudo
        magnitude_paragraph = document.add_paragraph(magnitude_descriptive_text)

        # Mengatur alignment paragraf menjadi justify
        magnitude_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY

        
        # Determine dominant depth category
        dominant_category = ''
        if percentage_depth_below_60 >= percentage_depth_60_to_300 and percentage_depth_below_60 >= percentage_depth_above_300:
            dominant_category = f'dangkal (<60 km) sebesar {percentage_depth_below_60}%'
        elif percentage_depth_60_to_300 >= percentage_depth_below_60 and percentage_depth_60_to_300 >= percentage_depth_above_300:
            dominant_category = f'kedalaman menengah (60 - 300 km) sebesar {percentage_depth_60_to_300}%'
        else:
            dominant_category = f'kedalaman dalam (>300 km) sebesar {percentage_depth_above_300}%'

        # Add space
        document.add_paragraph('')

        # Teks Deskriptif Kedalaman
        depth_descriptive_text = (
            f"Berdasarkan kedalaman, gempabumi yang mendominasi adalah kejadian gempabumi {dominant_category}  dari total kejadian gempabumi. "
            f"Jumlah gempabumi dangkal sebanyak {count_depth_below_60} kejadian gempabumi, "
            f"terdapat sebanyak {count_depth_60_to_300} kejadian gempabumi dengan kedalaman menengah 60 km - 300 km atau {percentage_depth_60_to_300}% dari total kejadian gempabumi, "
        )

        # Conditional text for depth above 300 km
        if count_depth_above_300 > 0:
            depth_descriptive_text += (
                f"dan kejadian gempa dalam sejumlah {count_depth_above_300} kejadian atau {percentage_depth_above_300}% dari total kejadian gempabumi."
            )
        else:
            depth_descriptive_text += "dan tidak terdapat kejadian gempabumi dengan kedalaman dalam >300 km."

        depth_paragraph = document.add_paragraph(depth_descriptive_text)
        depth_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY   

        # Add space
        document.add_paragraph('')

        # Teks analisis yang diminta
        analisis_text1 = (
            f"Selama satu minggu terakhir, gempabumi yang terjadi di Bali dan sekitarnya dikelompokan berdasarkan sumbernya (Gambar 3):\n"
        )    

        # Menambahkan paragraf analisis ke dalam dokumen
        analisis_paragraph1 = document.add_paragraph(analisis_text1)
        analisis_paragraph1.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT

        analisis_text2 = (
            f"1. Sebanyak {southern_earthquake_count} kejadian gempabumi terjadi di selatan Pulau Jawa, Bali, dan Lombok "
            f"yang diperkirakan berasosiasi dengan subduksi Indo Australia-Eurasia.\n"
            f"2. Sebanyak {shallow_earthquake_count} kejadian gempabumi terjadi menyebar di wilayah Pulau Bali, Lombok dan sekitarnya "
            f"diperkirakan berkaitan dengan sesar di belakang busur Flores atau Flores Back Arc Thrust dan aktivitas sesar aktif."
        )

        # Menambahkan paragraf analisis ke dalam dokumen
        analisis_paragraph2 = document.add_paragraph(analisis_text2)
        analisis_paragraph2.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT

        # Menghitung jumlah kejadian gempa yang dirasakan
        earthquakes_felt = earthquake_data[earthquake_data['Dirasakan'].notna()]
        total_felt = len(earthquakes_felt)

        # Menyusun narasi pembuka
        if total_felt > 0:
            felt_intro = (
                f"Selama periode ini terdapat {total_felt} kejadian gempabumi yang dirasakan oleh masyarakat "
                f"di wilayah Bali dan sekitarnya."
            )
        else:
            felt_intro = (
                f"Selama periode ini tidak terdapat kejadian gempabumi yang dirasakan oleh masyarakat "
                f"di wilayah Bali dan sekitarnya."
            )

        # Menambahkan narasi pembuka ke dokumen
        felt_intro_paragraph = document.add_paragraph(felt_intro)
        felt_intro_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY    

        # Menyusun detail setiap kejadian gempa yang dirasakan dalam format list dengan nomor urut
        if total_felt > 0:
            # Inisialisasi counter untuk nomor urut
            counter = 1

            for index, row in earthquakes_felt.iterrows():
                magnitude = row['Magnitude']
                date = row['Tanggal'].strftime('%d %B %Y')  # Format tanggal DD/MM/YYYY
                time = row['Waktu (WIB)']  # Asumsikan kolom waktu sudah dalam format string yang benar
                location = row['Dirasakan']
                description = row['Keterangan']
                depth = row['Kedalaman']

                # Format detail gempa yang dirasakan dengan nomor urut
                felt_detail = (
                    f"{counter}. Gempabumi dengan magnitudo {magnitude} {location} yang terjadi pada tanggal {date} "
                    f"pukul {time} WIB berlokasi di {description} pada kedalaman {depth} km."
                )

                # Menambahkan detail gempa ke dalam dokumen sebagai paragraf terpisah
                felt_detail_paragraph = document.add_paragraph(felt_detail)
                felt_detail_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY

                # Meningkatkan counter untuk nomor urut berikutnya
                counter += 1
        else:
            # Jika tidak ada gempa yang dirasakan, tambahkan keterangan tambahan
            no_felt_detail = document.add_paragraph(" ")
            no_felt_detail.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

        # Add Magnitude Bar Chart Image
        try:
            document.add_paragraph().add_run().add_picture(magnitude_bar_chart_path, width=Inches(5.65),) 
            document.paragraphs[-1].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
        except FileNotFoundError:
            document.add_paragraph('DIAGRAM BATANG MAGNITUDO TIDAK DITEMUKAN').bold = True

        # Add Magnitude Pie Chart Image
        try:
            document.add_paragraph().add_run().add_picture(magnitude_pie_chart_path, width=Inches(3.15))
            document.paragraphs[-1].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
        except FileNotFoundError:
            document.add_paragraph('DIAGRAM LINGKARAN MAGNITUDO TIDAK DITEMUKAN').bold = True

        # Teks Keterangan Gambar Magnitudo
        document.add_paragraph('Gambar 1. Histogram gempabumi harian berdasarkan magnitudo (kiri) dan diagram gempabumi berdasarkan magnitudo (kanan)').alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

        # Add space
        document.add_paragraph('')

         # Add Depth Bar Chart Image
        try:
            document.add_paragraph().add_run().add_picture(depth_bar_chart_path, width=Inches(5.65))
            document.paragraphs[-1].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
        except FileNotFoundError:
            document.add_paragraph('DIAGRAM BATANG KEDALAMAN TIDAK DITEMUKAN').bold = True

        # Add Depth Pie Chart Image
        try:
            document.add_paragraph().add_run().add_picture(depth_pie_chart_path, width=Inches(3.15))
            document.paragraphs[-1].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
        except FileNotFoundError:
            document.add_paragraph('DIAGRAM LINGKARAN KEDALAMAN TIDAK DITEMUKAN').bold = True

        # Teks Keterangan Gambar Kedalaman (akan diisi sesuai format)
        document.add_paragraph('Gambar 2.  Histogram gempabumi harian berdasarkan kedalaman (kiri) dan diagram gempabumi berdasarkan kedalaman (kanan)').alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

        # Add space
        document.add_paragraph('')
        
        # Add Seismic Map Image
        try:
            document.add_paragraph().add_run().add_picture(seismic_map_path, width=Inches(6))
            document.paragraphs[-1].alignment = WD_PARAGRAPH_ALIGNMENT.CENTER
        except FileNotFoundError:
            document.add_paragraph('PETA SEISMISITAS TIDAK DITEMUKAN').bold = True

        # Teks Keterangan Gambar Peta Seismisitas
        peta_seismisitas_text = (
            f"Gambar 3. Peta seismisitas Bali dan sekitarnya periode {nama_folder}"
        )
        peta_seismisitas_paragraph = document.add_paragraph(peta_seismisitas_text)
        peta_seismisitas_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

        # Tambahkan section baru dengan orientasi landscape
        new_section = document.add_section(WD_SECTION.NEW_PAGE)
        new_section.orientation = WD_ORIENT.LANDSCAPE

        # Atur ukuran kertas agar landscape sesuai
        new_section.page_width, new_section.page_height = new_section.page_height, new_section.page_width

        # Menentukan margin untuk section baru agar tabel berada di tengah secara vertikal
        section_margin = Cm(2)  # Sesuaikan margin jika perlu
        new_section.top_margin = section_margin
        new_section.bottom_margin = section_margin
        new_section.left_margin = section_margin
        new_section.right_margin = section_margin

        # Menambahkan paragraf baru untuk membungkus tabel
        table_paragraph = document.add_paragraph()
        table_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER  # Mengatur alignment paragraf ke tengah

        # Membuat teks keterangan tabel gempabumi
        tabel_keterangan_text = (
            f"Tabel 1. Data gempabumi di Bali dan sekitarnya tanggal {nama_folder}"
        )

        # Menambahkan teks keterangan tabel ke dalam dokumen
        tabel_keterangan_paragraph = document.add_paragraph(tabel_keterangan_text)
        tabel_keterangan_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER  # Mengatur alignment menjadi tengah

        # Menambahkan tabel langsung ke dalam paragraf
        table = document.add_table(rows=1, cols=len(earthquake_data.columns))
        table.style = 'Table Grid'

        # Set table alignment to center
        table.alignment = WD_TABLE_ALIGNMENT.CENTER

        # Set column widths to match the text content
        column_widths = {
            "NO.": 1.5,  # NO
            "Tanggal": 2.5,  # TANGGAL
            "Waktu (WIB)": 2.5,  # WAKTU (WIB)
            "Lintang": 2.0,  # LATITUDE
            "Bujur": 2.0,  # LONGITUDE
            "Kedalaman": 2.0,  # Depth
            "Magnitude": 1.5,  # Magnitude
            "Keterangan": 5.0,  # Keterangan
            "Dirasakan": 2.5  # Dirasakan
        }

        # Add header row
        hdr_cells = table.rows[0].cells
        for i, column_name in enumerate(earthquake_data.columns):
            hdr_cells[i].text = column_name
            hdr_cells[i].width = Cm(column_widths.get(column_name, 2.0))  # Set width based on column

        # Add data rows, memformat kolom 'Tanggal' tanpa jam
        for index, row in earthquake_data.iterrows():
            row_cells = table.add_row().cells
            for i, value in enumerate(row):
                # Jika kolom adalah 'Tanggal', hanya tampilkan tanggal tanpa jam
                if earthquake_data.columns[i] == 'Tanggal':
                    row_cells[i].text = value.strftime('%d/%m/%Y')  # Format tanggal menjadi DD/MM/YYYY
                # Jika kolom adalah 'Dirasakan' dan nilai adalah NaN, ganti dengan teks yang sesuai
                elif earthquake_data.columns[i] == 'Dirasakan':
                    # Ganti NaN dengan string kosong atau teks lain
                    row_cells[i].text = '' if pd.isna(value) else str(value)
                else:
                    row_cells[i].text = str(value)

                # Set the width of each data cell, sesuai dengan kolom
                row_cells[i].width = Cm(column_widths.get(earthquake_data.columns[i], 2.0))

        # Mengatur tabel agar berada di tengah secara horizontal
        table.alignment = WD_TABLE_ALIGNMENT.CENTER
        
        
        # Simpan 'datalengkap.csv' ke folder baru dengan nama 'data gempa (nama folder).xlsx'
        output_file = os.path.join(nama_folder, f'Press Release Gempabumi Periode {nama_folder}.docx')
    
        # Save the document
        document.save(output_file)

    messagebox.showinfo("Cetak Laporan", "Berhasil Mencetak Press Release")

def print_description():
    # Set locale to Indonesian for date formatting
    locale.setlocale(locale.LC_TIME, 'id_ID.UTF-8')

    # Fungsi untuk memilih file
    def select_file():
        root = tk.Tk()
        root.withdraw()  # Menyembunyikan jendela utama
        file_path = filedialog.askopenfilename(title="Pilih file laporan_gempa.xlsx", filetypes=[("Excel files", "*.xlsx")])
        return file_path

    # Memilih file Excel
    file_path = select_file()
    if not file_path:
        print("Tidak ada file yang dipilih.")
    else:
        # Load the Excel file
        earthquake_data = pd.read_excel(file_path)

        # Pastikan kolom 'Tanggal' dalam format datetime
        earthquake_data['Tanggal'] = pd.to_datetime(earthquake_data['Tanggal'])

        # Ambil tanggal awal dan akhir
        start_date = tanggal_awal
        end_date = tanggal_akhir
        
        # Format tanggal untuk digunakan dalam nama folder     

        earthquake_data = pd.read_excel(file_path)

        # Pastikan kolom 'Tanggal' dalam format datetime
        earthquake_data['Tanggal'] = pd.to_datetime(earthquake_data['Tanggal'])

        # Tentukan nama folder berdasarkan tanggal awal dan akhir
        if tanggal_awal.month == tanggal_akhir.month:
            nama_folder = f"{tanggal_awal.day:02d} - {tanggal_akhir.day:02d} {tanggal_awal.strftime('%B %Y')}"
        else:
            nama_folder = f"{tanggal_awal.day:02d} {tanggal_awal.strftime('%B')} - {tanggal_akhir.day:02d} {tanggal_akhir.strftime('%B %Y')}"

        # Calculate the required values
        total_earthquakes = len(earthquake_data)
        min_magnitude = earthquake_data['Magnitude'].min()
        max_magnitude = earthquake_data['Magnitude'].max()

        # Format tanggal dengan nama bulan dalam bahasa Indonesia
        start_date_formatted = start_date.strftime('%d %B %Y')  # contoh format: 01 Januari 2024
        end_date_formatted = end_date.strftime('%d %B %Y')      # contoh format: 05 September 2024
        #date_formatted=date1.strftime('%d %B %Y')
        #print(date_formatted)

        # Menentukan tanggal awal bulan untuk memulai perhitungan minggu
        start_of_month = start_date.replace(day=1)

        # Hitung nomor minggu relatif terhadap awal bulan
        week_number = (start_date - start_of_month).days // 7 + 1

        # Format bulan dalam bahasa Indonesia
        bulan_formatted = start_date.strftime('%B')

        # Menghitung jumlah kejadian gempabumi total
        total_earthquakes = len(earthquake_data)

        # Menghitung jumlah gempa dengan lat < -9 dan kedalaman < 60 km
        shallow_earthquake_count = earthquake_data[(earthquake_data['Lintang'] < -9) & 
                                                   (earthquake_data['Kedalaman'] < 60)].shape[0]

        # Menghitung jumlah gempa di selatan Pulau Jawa, Bali, dan Lombok
        southern_earthquake_count = total_earthquakes - shallow_earthquake_count

        # Magnitude categories
        magnitude_below_3 = earthquake_data[earthquake_data['Magnitude'] <= 3]
        magnitude_3_to_5 = earthquake_data[(earthquake_data['Magnitude'] > 3) & (earthquake_data['Magnitude'] <= 5)]
        magnitude_above_5 = earthquake_data[earthquake_data['Magnitude'] > 5]

        count_below_3 = len(magnitude_below_3)
        count_3_to_5 = len(magnitude_3_to_5)
        count_above_5 = len(magnitude_above_5)

        percentage_below_3 = int((count_below_3 / total_earthquakes) * 100)
        percentage_3_to_5 = int((count_3_to_5 / total_earthquakes) * 100)
        percentage_above_5 = int((count_above_5 / total_earthquakes) * 100)

        # Magnitude categories
        depth_below_60 = earthquake_data[earthquake_data['Kedalaman'] < 60]
        depth_60_to_300 = earthquake_data[(earthquake_data['Kedalaman'] >= 60) & (earthquake_data['Kedalaman'] <= 300)]
        depth_above_300 = earthquake_data[earthquake_data['Kedalaman'] > 300]

        count_depth_below_60 = len(depth_below_60)
        count_depth_60_to_300 = len(depth_60_to_300)
        count_depth_above_300 = len(depth_above_300)

        percentage_depth_below_60 = int((count_depth_below_60 / total_earthquakes) * 100)
        percentage_depth_60_to_300 = int((count_depth_60_to_300 / total_earthquakes) * 100)
        percentage_depth_above_300 = int((count_depth_above_300 / total_earthquakes) * 100)

        # Determine dominant depth category
        dominant_category = ''
        if percentage_depth_below_60 >= percentage_depth_60_to_300 and percentage_depth_below_60 >= percentage_depth_above_300:
            dominant_category = f'dangkal (<60 km) sebesar {percentage_depth_below_60}%'
        elif percentage_depth_60_to_300 >= percentage_depth_below_60 and percentage_depth_60_to_300 >= percentage_depth_above_300:
            dominant_category = f'kedalaman menengah (60 - 300 km) sebesar {percentage_depth_60_to_300}%'
        else:
            dominant_category = f'kedalaman dalam (>300 km) sebesar {percentage_depth_above_300}%'

        # Create a new Word document
        document = Document()

        # Ukuran kertas F4 (21.0 cm x 33.0 cm)
        f4_width = Cm(21.0)
        f4_height = Cm(33.0)

        # Mengatur ukuran kertas F4 untuk setiap section di dokumen
        for section in document.sections:
            # Mengatur orientasi potrait untuk F4
            section.orientation = WD_ORIENT.PORTRAIT  # Bisa diubah ke WD_ORIENT.LANDSCAPE jika ingin landscape
            section.page_width = f4_width
            section.page_height = f4_height

            # Mengatur margin halaman (optional, bisa disesuaikan)
            section.top_margin = Cm(2.5)     # Marginn atas
            section.bottom_margin = Cm(2.5)  # Marginn bawah
            section.left_margin = Cm(2.5)    # Marginn kiri
            section.right_margin = Cm(2.5)   # Marginn kanan

        # Add Subtitle
        subtitle = document.add_paragraph()
        subtitle_run = subtitle.add_run('Informasi Gempabumi di Wilayah Bali dan Sekitarnya')
        subtitle_run.font.size = Pt(12)
        subtitle.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

        # Add Date Period
        period = document.add_paragraph()
        period_run = period.add_run(f'Periode {nama_folder}')
        period_run.font.size = Pt(12)
        period.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

        # Add space
        document.add_paragraph('')

        # Teks Deskriptif Magnitudo dengan perhitungan minggu bulanan
        magnitude_descriptive_text = (
            f"Berdasarkan data Stasiun Geofisika Denpasar selama minggu ke-{week_number} bulan "
            f"{bulan_formatted} {start_date.year}, "
            f"di daerah Bali dan sekitarnya telah terjadi {total_earthquakes} kejadian gempabumi dengan magnitudo bervariasi "
            f"mulai dari M {min_magnitude} sampai M {max_magnitude}. Kejadian gempabumi dengan magnitudo M<3 sejumlah "
            f"{count_below_3} kejadian atau {percentage_below_3}% dari total kejadian gempabumi. Sedangkan untuk M 3 – 5 "
            f"sejumlah {count_3_to_5} kejadian atau {percentage_3_to_5}% dari total kejadian gempabumi "
        )

        # Conditional text for magnitude above 5
        if count_above_5 > 0:
            magnitude_descriptive_text += (
                f"dan kejadian gempa M>5 sejumlah {count_above_5} kejadian atau {percentage_above_5}% dari total kejadian gempabumi."
            )
        else:
            magnitude_descriptive_text += "dan tidak terdapat kejadian gempabumi dengan magnitudo M>5."

        # Menambahkan paragraf dengan teks deskriptif magnitudo
        magnitude_paragraph = document.add_paragraph(magnitude_descriptive_text)

        # Mengatur alignment paragraf menjadi justify
        magnitude_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY

        # Determine dominant depth category
        dominant_category = ''
        if percentage_depth_below_60 >= percentage_depth_60_to_300 and percentage_depth_below_60 >= percentage_depth_above_300:
            dominant_category = f'dangkal (<60 km) sebesar {percentage_depth_below_60}%'
        elif percentage_depth_60_to_300 >= percentage_depth_below_60 and percentage_depth_60_to_300 >= percentage_depth_above_300:
            dominant_category = f'kedalaman menengah (60 - 300 km) sebesar {percentage_depth_60_to_300}%'
        else:
            dominant_category = f'kedalaman dalam (>300 km) sebesar {percentage_depth_above_300}%'

        # Teks Deskriptif Kedalaman
        depth_descriptive_text = (
            f"Berdasarkan kedalaman, gempabumi yang mendominasi adalah kejadian gempabumi {dominant_category}  dari total kejadian gempabumi. "
            f"Jumlah gempabumi dangkal sebanyak {count_depth_below_60} kejadian gempabumi, "
            f"terdapat sebanyak {count_depth_60_to_300} kejadian gempabumi dengan kedalaman menengah 60 km - 300 km atau {percentage_depth_60_to_300}% dari total kejadian gempabumi, "
        )

        # Conditional text for depth above 300 km
        if count_depth_above_300 > 0:
            depth_descriptive_text += (
                f"dan kejadian gempa dalam sejumlah {count_depth_above_300} kejadian atau {percentage_depth_above_300}% dari total kejadian gempabumi."
            )
        else:
            depth_descriptive_text += "dan tidak terdapat kejadian gempabumi dengan kedalaman dalam >300 km."

        depth_paragraph = document.add_paragraph(depth_descriptive_text)
        depth_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY

        # Menghitung jumlah kejadian gempa yang dirasakan
        earthquakes_felt = earthquake_data[earthquake_data['Dirasakan'].notna()]
        total_felt = len(earthquakes_felt)

        # Menyusun narasi pembuka
        if total_felt > 0:
            felt_intro = (
                f"Selama periode ini terdapat {total_felt} kejadian gempabumi yang dirasakan oleh masyarakat "
                f"di wilayah Bali dan sekitarnya."
            )
        else:
            felt_intro = (
                f"Selama periode ini tidak terdapat kejadian gempabumi yang dirasakan oleh masyarakat "
                f"di wilayah Bali dan sekitarnya."
            )

        # Menambahkan narasi pembuka ke dokumen
        felt_intro_paragraph = document.add_paragraph(felt_intro)
        felt_intro_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY

        # Menyusun detail setiap kejadian gempa yang dirasakan dalam format list dengan nomor urut
        if total_felt > 0:
            # Inisialisasi counter untuk nomor urut
            counter = 1

            for index, row in earthquakes_felt.iterrows():
                magnitude = row['Magnitude']
                date = row['Tanggal'].strftime('%d %B %Y')  # Format tanggal DD/MM/YYYY
                time = row['Waktu (WIB)']  # Asumsikan kolom waktu sudah dalam format string yang benar
                location = row['Dirasakan']
                description = row['Keterangan']
                depth = row['Kedalaman']

                # Format detail gempa yang dirasakan dengan nomor urut
                felt_detail = (
                    f"{counter}. Gempabumi dengan magnitudo {magnitude} {location} yang terjadi pada tanggal {date} "
                    f"pukul {time} WIB berlokasi di {description} pada kedalaman {depth} km."
                )

                # Menambahkan detail gempa ke dalam dokumen sebagai paragraf terpisah
                felt_detail_paragraph = document.add_paragraph(felt_detail)
                felt_detail_paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.JUSTIFY

                # Meningkatkan counter untuk nomor urut berikutnya
                counter += 1
        else:
            # Jika tidak ada gempa yang dirasakan, tambahkan keterangan tambahan
            no_felt_detail = document.add_paragraph("Tidak ada rincian gempabumi yang dirasakan selama periode ini.")
            no_felt_detail.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

        # Add space
        document.add_paragraph('')
        
        document.add_paragraph("Halo sobat BMKG Denpasar! Apa kabar? Semoga sehat ya🤗\n" 
                            f"Berikut kami informasikan aktivitas kegempaan wilayah Bali dan sekitarnya periode {nama_folder}."
                            "Pada minggu ini, aktivitas kegempaan didominasi oleh gempa magnitudo M < 3 dengan kedalaman dangkal < 60 km."
                            f"Pada minggu ini terdapat {total_earthquakes} kejadian gempabumi." \
                            "\n" \
                            "\n" \
                            f"Jumat, {end_date_formatted}.\n"
                            ".\n"
                            ".\n"
                            ".\n"
                            "Hello BMKG Denpasar friends! How are you? Hope you are healthy🤗\n" 
                            f"Here we provide information on earthquake activity in the Bali area and its surroundings for the period {nama_folder}."
                            "This week, seismic activity was dominated by earthquakes with a magnitude of M < 3 with a shallow depth of < 60 km."
                            f"This week there are {total_earthquakes} earthquake event. Greetings of health and enthusiasm." \
                            "\n" \
                            "\n" \
                            f"Friday, {end_date_formatted}.\n"
                            "#stageofdenpasar #dewata #dedikasi #wibawa #tanggap #akuntabel #siapuntukselamat #kukul #kentongan #linuh")

        # Add space
        document.add_paragraph('')
        
        document.add_paragraph("Halo sobat BMKG Denpasar! Apa kabar? Semoga sehat ya🤗.\n"
                            f"Berikut kami informasikan aktivitas kegempaan wilayah Bali dan sekitarnya periode {nama_folder}.\n"
                            "Salam sehat dan semangat. \n"
                            "#stageofdenpasar #dewata #dedikasi #wibawa #tanggap #akuntabel #siapuntukselamat ")


        # Add space
        document.add_paragraph('')

        # Tambah penerima
        document.add_paragraph("Yth.\n1. Ibu Deputi Bidang Geofisika\n"
                            "2. Bapak Direktur Gempabumi dan Tsunami\n"
                            "3. Bapak Direktur Seismologi Teknik, Geofisika Potensial dan Tanda Waktu\n"
                            "4. Bapak Kepala Balai Wilayah III\n")

        # Tambah salam pembuka
        document.add_paragraph("Dengan hormat,\nBerikut kami sampaikan sebagai berikut:")

        # Tambah poin pertama
        document.add_paragraph(f"1. Infografis, videografis dan data excel seismisitas wilayah Bali periode {nama_folder}. "
                            "Laporan lengkap kami kirimkan ke email: zona.potensi@gmail.com")

        # Tambah daftar link media sosial
        document.add_paragraph("\nBerikut terlampir link media sosial infografis dan videografis seismisitas wilayah Bali dan sekitarnya dari Stasiun Geofisika Denpasar:")

        social_links = [
            ("1. Link Instagram", "Videografis dan Infografis: https://www.instagram.com/bmkg_denpasar"),
            ("2. Link Facebook", "Videografis dan Infografis: https://web.facebook.com/BMKGDenpasar"),
            ("3. Link X", "Videografis dan Infografis: https://x.com/BMKG_Denpasar"),
        ]

        for title, link in social_links:
            p = document.add_paragraph()
            p.add_run(title).bold = True
            p.add_run("\n" + link)

        # Tambah penutup
        document.add_paragraph("\nDemikian, mohon berkenan dan terima kasih.")
        document.add_paragraph("Hormat kami,\nSalam Dewata,\n \nBMKG Stasiun Geofisika Denpasar")

        # Tambah kontak
        document.add_paragraph("\nKontak Stasiun Geofisika Denpasar:")
        contacts = [
            "📍 Alamat: Jl. Pulau Tarakan No. 1 Sanglah, Denpasar-Bali",
            "📞 Telepon: (0361) 226157",
            "🌐 Web: stageof-bali.bmkg.go.id",
            "📧 Email: stageof.denpasar@bmkg.go.id"
        ]

        for contact in contacts:
            document.add_paragraph(contact)

        # Tambah kalimat penutup
        document.add_paragraph()
        document.add_paragraph("Om Swastiastu, Selamat Pagi sobat BMKG. Berikut kami sampaikan informasi geofisika dari Stasiun Geofisika Denpasar yang dapat diakses melalui link berikut:")
        document.add_paragraph("\n https://linktr.ee/StageofDenpasar \n")
        document.add_paragraph("\n Informasi meliputi:")
        document.add_paragraph("\n ✔ Informasi Sambaran Petir Mingguan")
        document.add_paragraph("\n ✔ Informasi Waktu Terbit dan Tenggelam Matahari Mingguan")
        document.add_paragraph("\n ✔ Informasi Gempabumi Mingguan \n")
        document.add_paragraph("\n Demikian, mohon berkenan dan terima kasih. \n")
        document.add_paragraph("Hormat kami,\nSalam Dewata,\n \nBMKG Stasiun Geofisika Denpasar")

        # Tambah kontak
        document.add_paragraph("\nKontak Stasiun Geofisika Denpasar:")
        contacts = [
            "📍 Alamat: Jl. Pulau Tarakan No. 1 Sanglah, Denpasar-Bali",
            "📞 Telepon: (0361) 226157",
            "🌐 Web: stageof-bali.bmkg.go.id",
            "📧 Email: stageof.denpasar@bmkg.go.id"
        ]

        for contact in contacts:
            document.add_paragraph(contact)

       # Simpan 'datalengkap.csv' ke folder baru dengan nama 'data gempa (nama folder).xlsx'
        output_file = os.path.join(nama_folder, f'Deskripsi info seismisitas website {nama_folder}.docx')
    
        # Save the document
        document.save(output_file)
        
    messagebox.showinfo("Cetak Deskripsi", "Berhasil Mencetak deskripsi")

def create_output_popup(title, message, header_color):
    """Create a popup for output messages"""
    popup = tk.Toplevel(root)
    popup.title(title)
    popup.geometry("450x280")
    popup.configure(bg=COLOR_WHITE)
    popup.resizable(False, False)
    
    # Header
    header_frame = tk.Frame(popup, bg=header_color, height=50)
    header_frame.pack(fill=tk.X)
    header_frame.pack_propagate(False)
    header_label = tk.Label(header_frame, text=title, 
                           font=("Gill Sans MT", 14, "bold"), bg=header_color, fg=COLOR_WHITE)
    header_label.pack(pady=12)
    
    # Content
    content_frame = tk.Frame(popup, bg=COLOR_WHITE)
    content_frame.pack(fill=tk.BOTH, expand=True, padx=20, pady=20)
    
    msg_label = tk.Label(content_frame, text=message,
                        font=("Gill Sans MT", 12), bg=COLOR_WHITE, fg=COLOR_DARK_TEXT,
                        wraplength=380, justify=tk.CENTER)
    msg_label.pack(expand=True)
    
    # Close button
    close_btn = tk.Button(popup, text="Tutup", command=popup.destroy,
                         bg=header_color, fg=COLOR_WHITE,
                         font=("Gill Sans MT", 10, "bold"), padx=30, pady=10,
                         relief=tk.FLAT, cursor="hand2", bd=0)
    close_btn.pack(pady=15)
    
    return popup

def show_petunjuk():
    """Show editable petunjuk penggunaan in main area"""
    # Clear and show petunjuk
    pass

# ========== MAIN WINDOW ==========
root = tk.Tk()
root.title("Seismioto - Otomatisasi Informasi Gempabumi Mingguan")
root.geometry("700x400")
root.configure(bg=COLOR_WHITE)
root.resizable(True, True)
root.state('zoomed')

# ========== MAIN FRAME ==========
main_frame = tk.Frame(root, bg=COLOR_WHITE)
main_frame.pack(fill=tk.BOTH, expand=True)

# ========== HEADER ==========
header_frame = tk.Frame(main_frame, bg=COLOR_GREEN_PASTEL, height=80)
header_frame.pack(fill=tk.X, side=tk.TOP)
header_frame.pack_propagate(False)

title_font = tkFont.Font(family="Ink Free", size=25, weight="bold")
title_label = tk.Label(header_frame, text="🌍 Seismioto", 
                       font=title_font, bg=COLOR_GREEN_PASTEL, fg=COLOR_DARK_TEXT)
title_label.pack(pady=(12, 0))

subtitle_label = tk.Label(header_frame, text="Otomatisasi Informasi Gempabumi Mingguan",
                         font=("Gill Sans MT", 13), bg=COLOR_GREEN_PASTEL, fg=COLOR_DARK_TEXT)
subtitle_label.pack(pady=(0, 2))

# ========== TOP SECTION (Input Controls) ==========
top_section = tk.Frame(main_frame, bg=COLOR_WHITE, height=120)
top_section.pack(fill=tk.X, side=tk.TOP)
top_section.pack_propagate(False)

# --- Left: File Selection ---
file_frame = tk.Frame(top_section, bg=COLOR_WHITE, borderwidth=0, relief=tk.SOLID)
file_frame.place(x=10,y=10)

file_icon_label = tk.Label(file_frame, text="📂 Pilih File", 
                          font=("Gill Sans MT", 15, "bold"), bg=COLOR_WHITE, fg=COLOR_DARK_TEXT)
file_icon_label.pack(anchor=tk.W, padx=10, pady=10)

file_button_container = tk.Frame(file_frame, bg=COLOR_WHITE)
file_button_container.pack(anchor=tk.W, padx=10, pady=(0, 8))

file_button = tk.Button(file_button_container, text="Pilih File Gempa", command=select_files,
                       bg=COLOR_BLUE_PASTEL, fg=COLOR_DARK_TEXT,
                       font=("Gill Sans MT", 11, "bold"), padx=12, pady=6,
                       relief=tk.FLAT, cursor="hand2", bd=0)
file_button.pack(side=tk.LEFT, padx=(0, 8))

file_status_label = tk.Label(file_button_container, text="❌ File belum dipilih",
                            font=("Gill Sans MT", 9, "bold"), bg=COLOR_WHITE, fg="#C00000")
file_status_label.pack(side=tk.LEFT, padx=0)

# --- Middle: Date Selection ---
date_frame = tk.Frame(top_section, bg=COLOR_WHITE, borderwidth=0, relief=tk.SOLID)
date_frame.place(x=490,y=10)

date_icon_label = tk.Label(date_frame, text="📅 Pilih Rentang Tanggal", 
                          font=("Gill Sans MT", 15, "bold"), bg=COLOR_WHITE, fg=COLOR_DARK_TEXT)
date_icon_label.pack(anchor=tk.N, padx=10, pady=10)

date_picker_frame = tk.Frame(date_frame, bg=COLOR_WHITE)
date_picker_frame.pack(anchor=tk.W, padx=10, pady=10)

start_label = tk.Label(date_picker_frame, text="Tanggal Awal:", 
                      font=("Gill Sans MT", 12), bg=COLOR_WHITE, fg=COLOR_DARK_TEXT)
start_label.pack(side=tk.LEFT, padx=(0, 0))

start_date_entry = DateEntry(date_picker_frame, width=13, background=COLOR_BLUE_PASTEL, 
                            foreground=COLOR_WHITE, borderwidth=1, 
                            date_pattern='dd/mm/yyyy', year=datetime.now().year)
start_date_entry.pack(side=tk.LEFT, padx=(0, 0))

end_label = tk.Label(date_picker_frame, text="Tanggal Akhir:", 
                    font=("Gill Sans MT", 12), bg=COLOR_WHITE, fg=COLOR_DARK_TEXT)
end_label.pack(side=tk.LEFT, padx=(10, 0))

end_date_entry = DateEntry(date_picker_frame, width=13, background=COLOR_BLUE_PASTEL, 
                          foreground=COLOR_WHITE, borderwidth=1, 
                          date_pattern='dd/mm/yyyy', year=datetime.now().year)
end_date_entry.pack(side=tk.LEFT)

# --- Right: Process Button ---
process_frame = tk.Frame(top_section, bg=COLOR_WHITE, borderwidth=0, relief=tk.SOLID)
process_frame.place(x=1300,y=10)

process_button = tk.Button(process_frame, text="🟢 Proses Data", command=process_data,
                          bg=COLOR_GREEN_PASTEL, fg=COLOR_DARK_TEXT,
                          font=("Gill Sans MT", 15, "bold"), padx=30, pady=30,
                          relief=tk.FLAT, cursor="hand2", bd=0)
process_button.pack(padx=0, pady=0)

# ========== DIVIDER LINE ==========
divider = tk.Frame(main_frame, bg="#000000", height=1)
divider.pack(fill=tk.X, side=tk.TOP)

# ========== BOTTOM SECTION (Sidebar + Content) ==========
bottom_section = tk.Frame(main_frame, bg=COLOR_PETUNJUK_BG)
bottom_section.pack(fill=tk.BOTH, expand=True, side=tk.TOP)

# ========== LEFT SIDEBAR (Cetak Output Buttons) ==========
sidebar = tk.Frame(bottom_section, bg=COLOR_WHITE, width=180)
sidebar.pack(side=tk.LEFT, fill=tk.Y, padx=0, pady=0)
sidebar.pack_propagate(False)

# Sidebar title
sidebar_title = tk.Label(sidebar, text="⚙ Cetak Output", 
                        font=("Gill Sans MT", 11, "bold"), bg=COLOR_WHITE, fg=COLOR_DARK_TEXT)
sidebar_title.pack(pady=12, padx=8, fill=tk.X)

# Output buttons with alternating colors
output_buttons_config = [
    ("Cetak Analisis XLSX", print_analysis, COLOR_BLUE_PASTEL),
    ("Cetak Peta", print_map, COLOR_GREEN_PASTEL),
    ("Cetak Laporan Word", print_report, COLOR_BLUE_PASTEL),
    ("Cetak Press Release", print_pressrelease, COLOR_GREEN_PASTEL),
    ("Cetak Deskripsi", print_description, COLOR_BLUE_PASTEL),
]

for text, command, color in output_buttons_config:
    btn = tk.Button(sidebar, text=text, command=command,
                   bg=color, fg=COLOR_DARK_TEXT,
                   font=("Gill Sans MT", 10,"bold"), padx=8, pady=30,
                   relief=tk.FLAT, cursor="hand2", bd=0,
                   wraplength=160, justify=tk.CENTER)
    btn.pack(fill=tk.X, padx=8, pady=5)

# Version label at bottom
version_label = tk.Label(sidebar, text="VER FDSN 1.0", 
                        font=("Gill Sans MT", 7), bg=COLOR_WHITE, fg=COLOR_DARK_TEXT)
version_label.pack(side=tk.BOTTOM, pady=8)

# ========== RIGHT CONTENT AREA (Petunjuk Penggunaan) ==========
content_area = tk.Frame(bottom_section, bg=COLOR_PETUNJUK_BG, borderwidth=0, relief=tk.SOLID)
content_area.pack(side=tk.LEFT, fill=tk.BOTH, expand=True, padx=10, pady=10)

# Content title
content_title = tk.Label(content_area, text="Petunjuk Penggunaan", 
                        font=("Gill Sans MT", 14, "bold"), bg=COLOR_PETUNJUK_BG, fg=COLOR_DARK_TEXT)
content_title.pack(anchor=tk.W, padx=15, pady=(15, 10))

# Scrollable text area
text_frame = tk.Frame(content_area, bg=COLOR_WHITE)
text_frame.pack(fill=tk.BOTH, expand=True, padx=15, pady=(0, 15))

scrollbar = ttk.Scrollbar(text_frame)
scrollbar.pack(side=tk.RIGHT, fill=tk.Y)

petunjuk_text = tk.Text(text_frame, height=20, width=80,
                       bg=COLOR_LIGHT_BG, fg=COLOR_DARK_TEXT,
                       font=("Gill Sans MT", 10), relief=tk.FLAT,
                       borderwidth=0, yscrollcommand=scrollbar.set, wrap=tk.WORD)
petunjuk_text.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
scrollbar.config(command=petunjuk_text.yview)

# Default petunjuk text
default_text = """PANDUAN PENGGUNAAN SEISMIOTO

1. PILIH FILE
   - Klik tombol "📁 Pilih File" untuk memilih file data gempabumi
   - Pastikan format file sesuai dengan yang diminta (.xlsx, .csv, atau format lainnya)
   - Status akan ditampilkan setelah file dipilih

2. TENTUKAN RENTANG TANGGAL
   - Pilih tanggal awal dan tanggal akhir analisis
   - Gunakan kalender yang tersedia
   - Rentang tanggal akan memfilter data sesuai periode yang dipilih

3. PROSES DATA
   - Klik tombol "🟢 PROSES DATA" untuk memproses data
   - Tunggu hingga proses selesai (akan ada notifikasi)
   - Data akan diproses sesuai rentang tanggal yang dipilih

4. CETAK OUTPUT
   - Pilih jenis output yang ingin dicetak:
     • 📊 Cetak Analisis XLSX: Hasil analisis dalam format spreadsheet
     • 🗺️ Cetak Peta: Visualisasi spasial gempabumi
     • 📄 Cetak Laporan Word: Laporan lengkap dalam format dokumen
     • 📰 Cetak Press Release: Siaran pers resmi
     • 📝 Cetak Deskripsi: Deskripsi detail peristiwa gempabumi
   - Setiap tombol akan menghasilkan file output yang berbeda

5. EXPORT DAN PENYIMPANAN
   - Semua file output akan disimpan di folder yang telah ditentukan
   - Periksa folder output untuk mengakses hasil"""

petunjuk_text.insert(tk.END, default_text)
petunjuk_text.config(state=tk.NORMAL)

# ========== RUN APPLICATION ==========
root.mainloop()


<>:813: SyntaxWarning: invalid escape sequence '\s'
<>:813: SyntaxWarning: invalid escape sequence '\s'
C:\Users\Hype G12\AppData\Local\Temp\ipykernel_12156\2623635762.py:813: SyntaxWarning: invalid escape sequence '\s'
  cross = pd.read_csv('crsx.dat', delimiter='\s+', header=None)
